# Notebook 04 v26 — Certified Stage 1 High-Recall Sieve, Split-Domain Ensemble & Benchmark Suite

**Sequential Pipeline Overhaul — Step 4 of 5 (`Notebook 04`).**
Restart the kernel, then Run All. This notebook consumes the certified `nb03_history_v26_certified` pretraining handoff and executes the canonical **Stage 1 High-Recall Sieve** with all model hyperparameters frozen:

1. **Pre-Filter Point-in-Time Cross-Sectional Ranking:** Computes all within-date percentile ranks (`_cs_pct`) and DuPont operating-archetype features across the **entire historically eligible investment universe (`feature_engineering_eligible == True`) on `as_of_date`** *before* applying future 12-month outcome completeness filters (`fold_membership_df`).
2. **Forward-Label Maturity & Holdout Boundary Verification:** Verifies that every training observation's `supervision_available_date` and `target_end_date_12m` mature strictly prior to `validation_start`, and that every validation observation matures prior to `HOLDOUT_START`.
3. **Warning-Free Numerical Stability (`newton-cholesky` Linear Rank Expert):** Eliminates Apple Silicon BLAS `matmul` overflow warnings in the L2 Linear Rank Expert via exact Cholesky Hessian steps on bounded `[-0.5, +0.5]` contiguous `float64` rank matrices, verified by an executable zero-warning assertion.
4. **Frozen Model Configurations & Full Benchmark Suite:** Evaluates the frozen `Legacy v24`, `v25 Market Only`, `v25 Fundamentals Only`, `Champion v25 Single Tree (116 cols)`, `Subspace-Bagged Tree (120 cols)`, and the **4-Expert Split-Domain Ensemble Champion** alongside continuous regression benchmarks (`Predict zero (VOO)` and `Predict training mean`).
5. **Pre-Cleanup vs. Post-Cleanup Baseline Comparison & Canonical Stage 1 Export:** Compares `v26 Certified` against the frozen `pre_cleanup_baseline_snapshot` on both raw folds and the exact intersection of observations, and writes one unified Stage 1 handoff (`stage1_recalled_candidates_for_llm.parquet`, `run_manifest.json`, and `output_sha256.json`).

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import warnings
import zipfile
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    fbeta_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from lightgbm import LGBMClassifier, LGBMRegressor

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE
DATA = PROJECT_ROOT / "data"
AUDIT_DIR = DATA / "interim/audits"

# Auto-discover latest certified NB03 v26 run
nb03_v26_root = AUDIT_DIR / "nb03_history_v26_certified"
if not nb03_v26_root.is_dir():
    raise FileNotFoundError(f"Run Notebook 03 v26 first. Missing: {nb03_v26_root}")
ARTIFACT_DIR = sorted([
    d for d in nb03_v26_root.iterdir()
    if d.is_dir() and (d / "pretraining_contract.json").is_file()
])[-1]

PRE_CLEANUP_SNAPSHOT_DIR = AUDIT_DIR / "pre_cleanup_baseline_snapshot"

# ==============================================================================
# FROZEN MODEL & SIEVE CONFIGURATION (DO NOT TUNE DURING INTEGRITY OVERHAUL)
# ==============================================================================
SEED = 42
HURDLE_EXCESS_RETURN = 0.10
STAGE1_TOP_FRACTION = 0.30

CLF_PARAMS = dict(
    objective="binary",
    n_estimators=250,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=100,
    reg_lambda=2.0,
    subsample_for_bin=50000,
    random_state=SEED,
    n_jobs=4,
    verbose=-1,
)

BAGGED_CLF_PARAMS = dict(
    CLF_PARAMS,
    n_estimators=350,
    learning_rate=0.025,
    num_leaves=15,
    min_child_samples=120,
    colsample_bytree=0.40,
    subsample=0.75,
    subsample_freq=1,
    reg_alpha=1.0,
    reg_lambda=5.0,
)

HUBER_REG_PARAMS = dict(
    objective="huber",
    alpha=0.25,
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=100,
    reg_lambda=2.0,
    random_state=SEED,
    n_jobs=4,
    verbose=-1,
)

pd.set_option("display.max_columns", 25)
ENVIRONMENT = {
    "python": platform.python_version(),
    "executable": sys.executable,
    "packages": {
        name: version(name)
        for name in ["numpy", "pandas", "pyarrow", "scikit-learn", "lightgbm"]
    },
}
print("Using certified NB03 v26 artifact dir:", ARTIFACT_DIR)
print(json.dumps(ENVIRONMENT, indent=2))

Using certified NB03 v26 artifact dir: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb03_history_v26_certified/20261103T152327935537Z
{
  "python": "3.12.7",
  "executable": "/opt/anaconda3/bin/python",
  "packages": {
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "pyarrow": "21.0.0",
    "scikit-learn": "1.5.2",
    "lightgbm": "4.6.0"
  }
}


## 1. Load & Verify the Certified `nb03_history_v26_certified` Contract and Checksums

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

contract_path = ARTIFACT_DIR / "pretraining_contract.json"
contract = json.loads(contract_path.read_text(encoding="utf-8"))
if contract.get("status") != "TECHNICAL_HANDOFF_ACCEPTED_FOR_ARCHITECTURE_RESEARCH":
    raise ValueError("Notebook 03 v26 technical handoff did not pass.")
checks = contract.get("technical_checks", {})
if not checks or not all(value is True for value in checks.values()):
    raise ValueError("Missing or failed Notebook 03 v26 technical checks.")

KEYS = contract["keys"]
FEATURE_COLUMNS = list(contract["feature_columns"])
V24_FEATURE_COLUMNS = list(contract.get("baseline_v24_feature_columns", []))
TARGET = contract["target"]
AVAILABLE = contract["availability_column"]
KIND = contract["outcome_kind_column"]

frame_path = ARTIFACT_DIR / contract["frame_file"]
fold_path = ARTIFACT_DIR / contract["fold_membership_file"]
pretraining_frame_df = pd.read_parquet(frame_path)
fold_membership_df = pd.read_csv(fold_path, dtype={"security_key": "string"})

loaded_artifacts = {p.name: sha256_file(p) for p in [contract_path, frame_path, fold_path]}
for path in [frame_path, fold_path]:
    expected = contract.get("artifact_sha256", {}).get(path.name)
    if not expected or sha256_file(path) != expected:
        raise ValueError(f"Artifact checksum missing or mismatched: {path.name}")

for df in [pretraining_frame_df, fold_membership_df]:
    df["security_key"] = df["security_key"].astype("string")
    df["as_of_date"] = pd.to_datetime(df["as_of_date"], errors="raise")
for col in [AVAILABLE, "target_end_date_12m"]:
    pretraining_frame_df[col] = pd.to_datetime(pretraining_frame_df[col], errors="raise")

holdout_dates = pretraining_frame_df.loc[
    pretraining_frame_df.research_partition.eq("sealed_holdout"), "as_of_date"
]
HOLDOUT_START = holdout_dates.min()

print(
    json.dumps(
        {
            "revision": contract["revision"],
            "total_panel_rows": len(pretraining_frame_df),
            "feature_eligible_rows": int(pretraining_frame_df["feature_engineering_eligible"].fillna(False).sum()),
            "active_v25_features": len(FEATURE_COLUMNS),
            "control_v24_features": len(V24_FEATURE_COLUMNS),
            "holdout_start": HOLDOUT_START.strftime("%Y-%m-%d"),
            "loaded_sha256": loaded_artifacts,
        },
        indent=2,
    )
)

{
  "revision": "nb03_v26_CERTIFIED_SCAR_TISSUE_VELOCITY",
  "total_panel_rows": 64194,
  "feature_eligible_rows": 59162,
  "active_v25_features": 109,
  "control_v24_features": 64,
  "holdout_start": "2024-08-30",
  "loaded_sha256": {
    "pretraining_contract.json": "2d7b30afdb4a0eeecb895e4b2314f6aa92153357b82dba8692de6d65b7720d3e",
    "pretraining_frame.parquet": "690b99415672efd0290c3ef55410054f8f4d9456265bb807022df7866b9e53ec",
    "pretraining_fold_membership.csv": "2e625b617b1370403cc0aac843eeb6c2aee0e1e29c26674c71857fb152247079"
  }
}


## 2. Compute Cross-Sectional Ranks on the Historically Eligible Investment Universe *Before* Label Filtering

To guarantee that a stock's within-date percentile rank ($\hat{F}_t(x_{i,t})$) on `as_of_date` $t$ depends **only** on the stocks that were investable and feature-eligible on date $t$—and **never** on whether other stocks on date $t$ survived the next 12 months with an `exact` return label:
1. We filter `pretraining_frame_df` to `elig_mask = pretraining_frame_df['feature_engineering_eligible'].fillna(False)`.
2. We compute all 12 `_cs_pct` features plus the 4 proven `v26` DuPont archetype/bulwark features (`fund_peer_archetype_fcf_rank`, `fund_productive_capex_compounder`, `fund_cyclical_cash_bulwark`, `fund_peer_archetype_rev_share_gain`) and `fund_fortress_quality_cs_score` grouped strictly by **`as_of_date`** on `elig_mask`.
3. Only **after** all point-in-time cross-sectional ranks are locked on `pretraining_frame_df` do we merge `fold_membership_df` and verify the forward-label maturity cutoffs.

In [3]:
CS_RANK_SOURCE_COLS = [
    "price_drawdown_from_252d_high",
    "price_ulcer_index_12m",
    "price_underwater_duration_20pct_12m",
    "price_vol_term_structure_126d_252d",
    "price_downside_vol_ratio_252d",
    "price_momentum_12m_ex_1m",
    "price_return_curvature_63d_vs_252d",
    "price_martin_ratio_252d",
    "fund_fcf_margin_ttm",
    "fund_roa_ttm",
    "fund_cash_to_assets_change_yoy",
    "fund_drawdown_x_cash_burn",
]

PROVEN_V26_COLS = [
    "fund_peer_archetype_fcf_rank",
    "fund_productive_capex_compounder",
    "fund_cyclical_cash_bulwark",
    "fund_peer_archetype_rev_share_gain",
]

def compute_pre_filter_cross_sectional_features(frame_in: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Computes all within-date cross-sectional percentile ranks strictly across
    `feature_engineering_eligible == True` rows on each `as_of_date`, BEFORE
    filtering for future 12-month outcome completeness.
    """
    df = frame_in.copy()
    elig_mask = df["feature_engineering_eligible"].fillna(False)
    elig_df = df.loc[elig_mask].copy()
    grp = elig_df.groupby("as_of_date", sort=False)

    cs_cols = []
    for col in CS_RANK_SOURCE_COLS:
        rk_col = f"{col}_cs_pct"
        elig_df[rk_col] = grp[col].rank(pct=True, ascending=True, na_option="keep").astype("float64")
        cs_cols.append(rk_col)

    def cs_rk(col_name: str, ascending: bool = True) -> pd.Series:
        return grp[col_name].rank(pct=True, ascending=ascending, na_option="keep").astype("float64")

    r_fcf_m = cs_rk("fund_fcf_margin_ttm", True)
    r_gm = cs_rk("fund_gross_margin_ttm", True)
    r_op_m = cs_rk("fund_operating_margin_ttm", True)
    r_roa = cs_rk("fund_roa_ttm", True)
    r_capex = cs_rk("fund_capex_intensity_ttm", True)
    r_gp_g = cs_rk("fund_gross_profit_growth_yoy", True)
    r_cash = cs_rk("fund_cash_to_assets", True)
    r_cash_chg = cs_rk("fund_cash_to_assets_change_yoy", True)
    r_op_m_chg = cs_rk("fund_operating_margin_change_yoy", True)
    r_debt = cs_rk("fund_debt_to_assets", True)
    r_delever = cs_rk("fund_debt_to_assets_change_yoy", False)
    r_low_sbc = cs_rk("fund_sbc_to_revenue_ttm", False)
    r_low_acc = cs_rk("fund_accruals_to_assets_ttm", False)

    # 1. Within-date DuPont Operating Archetype (Gross Margin quintiles on eligible universe)
    elig_df["_dupont_arch"] = (
        grp["fund_gross_margin_ttm"]
        .transform(lambda s: pd.qcut(s.rank(method="first"), q=5, labels=False, duplicates="drop"))
        .fillna(-1)
    )
    grp_arch = elig_df.groupby(["as_of_date", "_dupont_arch"], sort=False)
    elig_df["fund_peer_archetype_fcf_rank"] = (
        grp_arch["fund_fcf_margin_ttm"].rank(pct=True, ascending=True, na_option="keep").astype("float64")
    )
    peer_med_rev_g = grp_arch["fund_revenue_growth_yoy"].transform("median")
    elig_df["fund_peer_archetype_rev_share_gain"] = (
        elig_df["fund_revenue_growth_yoy"] - peer_med_rev_g
    ).astype("float64")

    # 2. Cyclical Cash Bulwark & Productive CapEx Compounder
    elig_df["fund_cyclical_cash_bulwark"] = (
        (1.0 - r_gm) * r_debt * (((r_fcf_m + r_cash + r_delever) / 3.0) - 0.50)
    ).astype("float64")
    reinvest_payoff = pd.concat([r_roa, r_gp_g, r_op_m_chg], axis=1).mean(axis=1)
    elig_df["fund_productive_capex_compounder"] = (
        r_capex * (reinvest_payoff - 0.50)
    ).astype("float64")

    # 3. Uniformly re-ranked Fortress Quality Score for Stage 3 Validity Anchors
    elig_df["_raw_q_mean"] = pd.concat(
        [r_fcf_m, r_op_m, r_roa, r_cash, r_cash_chg, r_op_m_chg, r_delever, r_low_sbc, r_low_acc],
        axis=1,
    ).mean(axis=1)
    elig_df["fund_fortress_quality_cs_score"] = (
        grp["_raw_q_mean"].rank(pct=True, ascending=True, na_option="keep").astype("float64")
    )

    new_rank_cols = cs_cols + PROVEN_V26_COLS + ["fund_fortress_quality_cs_score"]
    for c in new_rank_cols:
        df[c] = np.nan
        df.loc[elig_mask, c] = elig_df[c]

    return df, cs_cols

pretraining_frame_df, cs_rank_cols = compute_pre_filter_cross_sectional_features(pretraining_frame_df)

V25_WITH_CS_COLUMNS = list(dict.fromkeys(FEATURE_COLUMNS + cs_rank_cols))
RAW_COLLINEAR_PROXIES_TO_DROP = {
    "price_idiosyncratic_vol_252d",
    "price_drawdown_from_252d_high",
    "price_total_return_126d",
    "price_total_return_63d",
    "price_beta_voo_252d",
}
CHAMPION_116_COLS = [c for c in V25_WITH_CS_COLUMNS if c not in RAW_COLLINEAR_PROXIES_TO_DROP]
ENSEMBLE_ALL_COLS = list(dict.fromkeys(CHAMPION_116_COLS + PROVEN_V26_COLS))
ENSEMBLE_MKT_COLS = [c for c in ENSEMBLE_ALL_COLS if c.startswith("price_")]
ENSEMBLE_FUND_COLS = [c for c in ENSEMBLE_ALL_COLS if c.startswith("fund_")]
ALL_MODEL_COLUMNS = sorted(
    set(V25_WITH_CS_COLUMNS) | set(V24_FEATURE_COLUMNS) | set(ENSEMBLE_ALL_COLS) | {"fund_fortress_quality_cs_score"}
)

# Join fold_membership_df AFTER pre-filter cross-sectional ranks are locked
fold_rows_df = fold_membership_df.rename(columns={KIND: "membership_kind"}).merge(
    pretraining_frame_df, on=KEYS, how="left", validate="many_to_one", indicator=True
)
if not fold_rows_df["_merge"].eq("both").all():
    raise ValueError("Fold keys absent from frame.")

# Enforce forward-label maturity check on every fold and role
for fold_id, rows in fold_rows_df.groupby("fold", sort=False):
    validation_start = rows.loc[rows.role.eq("validation"), "as_of_date"].min()
    for role, subset in rows.groupby("role"):
        cutoff = validation_start if role == "train" else HOLDOUT_START
        if not ((subset[AVAILABLE] < cutoff) & (subset.target_end_date_12m < cutoff)).all():
            raise ValueError(f"Fold {fold_id}/{role}: outcome crosses its availability cutoff.")
        if subset["research_partition"].eq("sealed_holdout").any():
            raise ValueError(f"Fold {fold_id}/{role}: sealed holdout entered development fold.")

fold_rows_df = fold_rows_df.drop(columns=["_merge", "membership_kind"])
FOLD_IDS = fold_membership_df["fold"].drop_duplicates().tolist()

def get_fold(fold_id, hurdle=HURDLE_EXCESS_RETURN):
    if fold_id not in FOLD_IDS:
        raise KeyError(f"Choose one of {FOLD_IDS}")
    result = {}
    for role in ["train", "validation"]:
        rows = fold_rows_df.loc[fold_rows_df.fold.eq(fold_id) & fold_rows_df.role.eq(role)].copy()
        rows = rows.set_index(KEYS).sort_index()
        exact_rows = rows.loc[rows[KIND].eq("exact")]
        y_cont = exact_rows[TARGET].astype("float64").copy()
        y_bin = (y_cont >= hurdle).astype("int32").rename(f"beat_voo_by_{int(hurdle*100)}pct")
        result[role] = {
            "X": exact_rows.loc[:, ALL_MODEL_COLUMNS].copy(),
            "y": y_cont,
            "y_bin": y_bin,
            "metadata": exact_rows.drop(columns=ALL_MODEL_COLUMNS).copy(),
            "conditional_rows": rows.loc[rows[KIND].eq("conditional_interval")].copy(),
        }
    return result

# Audit how many extra eligible stocks per date were included in the pre-filter CS rank denominator
elig_counts = pretraining_frame_df.loc[pretraining_frame_df["feature_engineering_eligible"].fillna(False)].groupby("as_of_date").size()
exact_counts = pretraining_frame_df.loc[pretraining_frame_df["exact_supervision_candidate"].fillna(False)].groupby("as_of_date").size()
rank_denom_diff = (elig_counts - exact_counts).dropna()
print(f"Pre-filter CS ranking active: mean eligible stocks/date = {elig_counts.mean():.1f} vs. exact-label stocks/date = {exact_counts.mean():.1f} "
      f"(mean +{rank_denom_diff.mean():.1f} non-surviving/incomplete-outcome stocks included in date-t rank denominator!).")

Pre-filter CS ranking active: mean eligible stocks/date = 465.8 vs. exact-label stocks/date = 461.3 (mean +1.4 non-surviving/incomplete-outcome stocks included in date-t rank denominator!).


## 3. Frozen Monotonicity Priors, Warning-Free Linear Expert (`newton-cholesky`) & Evaluation Harness

In [4]:
# === REPLACEMENT FOR CELL 3 (In [4] | id=nb04-v26-007) ===
STRICT_MONOTONE_MAP = {
    "price_ulcer_index_6m": -1,
    "price_ulcer_index_12m": -1,
    "price_ulcer_index_12m_cs_pct": -1,
    "price_underwater_duration_20pct_12m": -1,
    "price_underwater_duration_20pct_12m_cs_pct": -1,
    "price_underwater_duration_40pct_12m": -1,
    "price_drawdown_from_252d_high_cs_pct": 1,
    "price_excess_drawdown_vs_voo_252d": 1,
    "price_acute_vs_chronic_drawdown": -1,
    "price_vol_term_structure_63d_252d": -1,
    "price_vol_term_structure_126d_252d": -1,
    "price_vol_term_structure_126d_252d_cs_pct": -1,
    "price_downside_vol_ratio_252d": -1,
    "price_downside_vol_ratio_252d_cs_pct": -1,
    "price_rel_vol_to_voo_252d": -1,
    "price_idio_vol_ratio_252d": -1,
    "fund_debt_to_assets": -1,
    "fund_debt_to_assets_change_yoy": -1,
    "fund_debt_growth_yoy": -1,
    "fund_net_debt_to_ebitda_ttm": -1,
    "fund_net_debt_to_assets": -1,
    "fund_debt_vs_cash_growth_spread": -1,
    "fund_sbc_to_revenue_ttm": -1,
    "fund_diluted_share_growth_yoy": -1,
    "fund_shareholder_yield_dilution_component": 1,
    "fund_cash_to_assets_change_yoy": 1,
    "fund_cash_to_assets_change_yoy_cs_pct": 1,
    "fund_drawdown_x_cash_burn": 1,
    "fund_drawdown_x_cash_burn_cs_pct": 1,
    "fund_drawdown_x_ocf_accel": 1,
    "fund_drawdown_x_margin_trajectory": 1,
    "price_drawdown_x_63d_stabilization": 1,
    "price_drawdown_x_21d_stabilization": 1,
    "price_return_curvature_63d_vs_252d": 1,
    "price_return_curvature_63d_vs_252d_cs_pct": 1,
    "fund_peer_archetype_fcf_rank": 1,
    "fund_productive_capex_compounder": 1,
    "fund_cyclical_cash_bulwark": 1,
    "fund_peer_archetype_rev_share_gain": 1,
}


def build_monotone_vector(
    columns: list[str], mono_map: dict = STRICT_MONOTONE_MAP
) -> list[int]:
    return [mono_map.get(col, 0) for col in columns]


def evaluate_stage1_sieve(
    y_cont: pd.Series,
    y_bin: pd.Series,
    prob_scores: pd.Series,
    top_fraction: float = STAGE1_TOP_FRACTION,
) -> dict:
    if not y_cont.index.equals(prob_scores.index):
        raise ValueError(
            "Index mismatch between outcomes and predicted probabilities."
        )
    if not np.isfinite(prob_scores.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite predicted probability scores detected!")

    df = pd.DataFrame(
        {
            "actual_excess": y_cont.to_numpy(dtype=float),
            "y_bin": y_bin.to_numpy(dtype=int),
            "prob": prob_scores.to_numpy(dtype=float),
        },
        index=y_cont.index,
    )

    df["monthly_pct_rank"] = df.groupby(level="as_of_date")["prob"].rank(
        pct=True, ascending=True
    )
    df["recalled_top_k"] = df["monthly_pct_rank"] >= (1.0 - top_fraction)

    recalled = df.loc[df["recalled_top_k"]]
    pruned = df.loc[~df["recalled_top_k"]]

    base_rate = float(df["y_bin"].mean())
    prec_top_k = float(
        precision_score(df["y_bin"], df["recalled_top_k"], zero_division=0)
    )
    rec_top_k = float(
        recall_score(df["y_bin"], df["recalled_top_k"], zero_division=0)
    )
    f2_top_k = float(
        fbeta_score(
            df["y_bin"], df["recalled_top_k"], beta=2.0, zero_division=0
        )
    )

    return {
        "ROC_AUC": float(roc_auc_score(df["y_bin"], df["prob"])),
        "PR_AUC": float(average_precision_score(df["y_bin"], df["prob"])),
        "Base_Rate_10pct": base_rate,
        f"Recall@Top{int(top_fraction*100)}%": rec_top_k,
        f"Precision@Top{int(top_fraction*100)}%": prec_top_k,
        "Precision_Lift": prec_top_k / max(base_rate, 1e-9),
        f"F2_Score@Top{int(top_fraction*100)}%": f2_top_k,
        "Recalled_Mean_Excess": float(recalled["actual_excess"].mean()),
        "Pruned_Mean_Excess": float(pruned["actual_excess"].mean()),
        "Sieve_Spread_vs_Pruned": float(
            recalled["actual_excess"].mean() - pruned["actual_excess"].mean()
        ),
        "Universe_Mean_Excess": float(df["actual_excess"].mean()),
        "Recalled_Beat_VOO_Rate": float(
            (recalled["actual_excess"] > 0.0).mean()
        ),
        "Recalled_Tail_Blowup_Rate(<-40%)": float(
            (recalled["actual_excess"] <= -0.40).mean()
        ),
        "Pruned_Tail_Blowup_Rate(<-40%)": float(
            (pruned["actual_excess"] <= -0.40).mean()
        ),
    }


def fit_predict_linear_rank_expert_stable(
    X_tr: pd.DataFrame,
    y_tr_bin: pd.Series,
    X_va: pd.DataFrame,
    C: float = 0.05,
) -> tuple[dict, pd.Series, list[str]]:
    """Warning-free, full-rank L2 Logistic Regression Expert (C=0.05 -> lambda = 20.0):

    1. Removes redundant `_cs_pct` duplicate columns when the raw source column
    is already present (since rank(col) == rank(col_cs_pct)) and drops exact
    negation duplicates.
    2. Solves the exact L2-penalized logistic regression objective via damped
    Newton-Raphson (IRLS) using `np.einsum` and LAPACK `np.linalg.solve` (which
    avoids the macOS Apple Silicon Accelerate `matmul` AMX status-register
    warning bug).
    3. Verifies zero RuntimeWarnings and confirms gradient convergence (||grad||_inf < 1e-8).
    """
    col_set = set(X_tr.columns)
    dedup_cols = [
        c
        for c in X_tr.columns
        if not (c.endswith("_cs_pct") and c[:-7] in col_set)
        and c != "fund_shareholder_yield_dilution_component"
    ]

    X_tr_rk = (
        X_tr.loc[:, dedup_cols]
        .groupby(level="as_of_date")
        .rank(pct=True)
        .fillna(0.50)
        - 0.50
    ).clip(-0.50, 0.50)
    X_va_rk = (
        X_va.loc[:, dedup_cols]
        .groupby(level="as_of_date")
        .rank(pct=True)
        .fillna(0.50)
        - 0.50
    ).clip(-0.50, 0.50)

    valid_cols = [c for c in dedup_cols if float(X_tr_rk[c].std()) > 1e-6]
    X_tr_mat = np.ascontiguousarray(
        X_tr_rk.loc[:, valid_cols].to_numpy(dtype=np.float64)
    )
    X_va_mat = np.ascontiguousarray(
        X_va_rk.loc[:, valid_cols].to_numpy(dtype=np.float64)
    )
    y_arr = np.ascontiguousarray(y_tr_bin.to_numpy(dtype=np.float64))

    # Append unpenalized intercept column
    n_tr, n_feat = X_tr_mat.shape
    X_tr_aug = np.column_stack([X_tr_mat, np.ones(n_tr, dtype=np.float64)])
    X_va_aug = np.column_stack(
        [X_va_mat, np.ones(X_va_mat.shape[0], dtype=np.float64)]
    )

    l2_penalty = 1.0 / float(C)
    reg_diag = np.full(n_feat + 1, l2_penalty, dtype=np.float64)
    reg_diag[-1] = 0.0  # Do not penalize intercept

    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", RuntimeWarning)
        theta = np.zeros(n_feat + 1, dtype=np.float64)
        p_bar = float(np.clip(y_arr.mean(), 1e-4, 1.0 - 1e-4))
        theta[-1] = np.log(p_bar / (1.0 - p_bar))

        for _ in range(25):
            eta = np.clip(np.einsum("ij,j->i", X_tr_aug, theta), -30.0, 30.0)
            p = 1.0 / (1.0 + np.exp(-eta))
            grad = np.einsum("ij,i->j", X_tr_aug, p - y_arr) + reg_diag * theta
            if float(np.max(np.abs(grad))) < 1e-8:
                break
            w = np.clip(p * (1.0 - p), 1e-8, 0.25)
            hess = np.einsum("ij,i,ik->jk", X_tr_aug, w, X_tr_aug) + np.diag(
                reg_diag
            )
            step = np.linalg.solve(hess, grad)
            theta = theta - step
            if float(np.max(np.abs(step))) < 1e-8:
                break

        val_logits = np.clip(np.einsum("ij,j->i", X_va_aug, theta), -30.0, 30.0)
        probs = 1.0 / (1.0 + np.exp(-val_logits))
        runtime_warns = [
            str(w.message)
            for w in caught_warnings
            if issubclass(w.category, RuntimeWarning)
        ]

    if runtime_warns:
        raise RuntimeError(
            f"RuntimeWarning in Linear Rank Expert: {runtime_warns}"
        )
    if float(np.max(np.abs(grad))) >= 1e-5:
        raise RuntimeError(
            f"Linear Rank Expert failed to converge: max_grad={np.max(np.abs(grad))}"
        )
    if not np.isfinite(probs).all():
        raise ValueError("Non-finite linear expert probabilities!")

    model_info = {
        "features": valid_cols,
        "coef": theta[:-1],
        "intercept": float(theta[-1]),
        "max_grad_norm": float(np.max(np.abs(grad))),
    }
    return (
        model_info,
        pd.Series(probs, index=X_va.index, name="linear_expert_prob"),
        runtime_warns,
    )


print(
    "Initialized frozen monotonicity vector and warning-free Newton-IRLS Linear Rank Expert."
)

Initialized frozen monotonicity vector and warning-free Newton-IRLS Linear Rank Expert.


## 4. Full Stage 1 Model Suite & Continuous Regression Benchmarks (`VOO` & `Training Mean`)

In [5]:
CANONICAL_STAGE1_MODEL = "6. 4-Expert Split-Domain Ensemble Champion (Frozen)"
LEGACY_MODEL_NAME = "1. Legacy v24 (64 cols, Unconstrained)"

market_v25 = [c for c in V25_WITH_CS_COLUMNS if c.startswith(("price_", "market_"))]
fundamental_v25 = [c for c in V25_WITH_CS_COLUMNS if c.startswith("fund_")]

ablation_rows = []
reg_comparison_rows = []
stage1_probabilities = {}
canonical_queues = []
runtime_warning_count = 0

def to_cs_rank(s: pd.Series) -> pd.Series:
    return s.groupby(level="as_of_date").rank(pct=True)

for fold_id in FOLD_IDS:
    fdata = get_fold(fold_id)
    X_tr_all = fdata["train"]["X"]
    y_tr_cont = fdata["train"]["y"]
    y_tr_bin = fdata["train"]["y_bin"]
    X_va_all = fdata["validation"]["X"]
    y_va_cont = fdata["validation"]["y"]
    y_va_bin = fdata["validation"]["y_bin"]

    # 1. Legacy v24 (64 cols, Unconstrained)
    clf_v24 = LGBMClassifier(**CLF_PARAMS)
    clf_v24.fit(X_tr_all.loc[:, V24_FEATURE_COLUMNS], y_tr_bin)
    p_v24 = pd.Series(clf_v24.predict_proba(X_va_all.loc[:, V24_FEATURE_COLUMNS])[:, 1], index=X_va_all.index)

    # 2. v25 Market/Scar-Tissue Only (Unconstrained diagnostic)
    clf_m_raw = LGBMClassifier(**CLF_PARAMS)
    clf_m_raw.fit(X_tr_all.loc[:, market_v25], y_tr_bin)
    p_m_raw = pd.Series(clf_m_raw.predict_proba(X_va_all.loc[:, market_v25])[:, 1], index=X_va_all.index)

    # 3. v25 Fundamentals + Velocity Only (Unconstrained diagnostic)
    clf_f_raw = LGBMClassifier(**CLF_PARAMS)
    clf_f_raw.fit(X_tr_all.loc[:, fundamental_v25], y_tr_bin)
    p_f_raw = pd.Series(clf_f_raw.predict_proba(X_va_all.loc[:, fundamental_v25])[:, 1], index=X_va_all.index)

    # 4. Champion v25 Single Tree (116 cols, Strict Monotone, colsample=1.0)
    clf_champ116 = LGBMClassifier(**CLF_PARAMS, monotone_constraints=build_monotone_vector(CHAMPION_116_COLS))
    clf_champ116.fit(X_tr_all.loc[:, CHAMPION_116_COLS], y_tr_bin)
    p_champ116 = pd.Series(clf_champ116.predict_proba(X_va_all.loc[:, CHAMPION_116_COLS])[:, 1], index=X_va_all.index)

    # 5 & 6. Frozen 4-Expert Split-Domain Ensemble Components (on 120 ENSEMBLE_ALL_COLS)
    X_tr_ens = X_tr_all.loc[:, ENSEMBLE_ALL_COLS]
    X_va_ens = X_va_all.loc[:, ENSEMBLE_ALL_COLS]
    mono_all = build_monotone_vector(ENSEMBLE_ALL_COLS)
    mono_mkt = build_monotone_vector(ENSEMBLE_MKT_COLS)
    mono_fund = build_monotone_vector(ENSEMBLE_FUND_COLS)

    # Expert 1 (45%): Subspace-Bagged Cross-Domain Tree
    clf_bag = LGBMClassifier(**BAGGED_CLF_PARAMS, monotone_constraints=mono_all)
    clf_bag.fit(X_tr_ens, y_tr_bin)
    p_bag = pd.Series(clf_bag.predict_proba(X_va_ens)[:, 1], index=X_va_ens.index)

    # Expert 2 (30%): Dedicated Market/Scar-Tissue Specialist Tree
    clf_mkt = LGBMClassifier(**dict(BAGGED_CLF_PARAMS, colsample_bytree=0.60, monotone_constraints=mono_mkt))
    clf_mkt.fit(X_tr_ens.loc[:, ENSEMBLE_MKT_COLS], y_tr_bin)
    p_mkt = pd.Series(clf_mkt.predict_proba(X_va_ens.loc[:, ENSEMBLE_MKT_COLS])[:, 1], index=X_va_ens.index)

    # Expert 3 (15%): Dedicated Fundamental/Archetype/Velocity Specialist Tree
    clf_fund = LGBMClassifier(**dict(BAGGED_CLF_PARAMS, colsample_bytree=0.50, monotone_constraints=mono_fund))
    clf_fund.fit(X_tr_ens.loc[:, ENSEMBLE_FUND_COLS], y_tr_bin)
    p_fund = pd.Series(clf_fund.predict_proba(X_va_ens.loc[:, ENSEMBLE_FUND_COLS])[:, 1], index=X_va_ens.index)

    # Expert 4 (10%): Warning-Free L2 Linear Rank Expert (newton-cholesky)
    lin_clf, p_lin, lin_warns = fit_predict_linear_rank_expert_stable(X_tr_ens, y_tr_bin, X_va_ens)
    runtime_warning_count += len(lin_warns)

    p_ens_4 = (
        0.45 * to_cs_rank(p_bag)
        + 0.30 * to_cs_rank(p_mkt)
        + 0.15 * to_cs_rank(p_fund)
        + 0.10 * to_cs_rank(p_lin)
    )

    for label, n_feat, prob_s in [
        (LEGACY_MODEL_NAME, len(V24_FEATURE_COLUMNS), p_v24),
        ("2. v25 Market/Scar-Tissue Only", len(market_v25), p_m_raw),
        ("3. v25 Fundamentals + Velocity Only", len(fundamental_v25), p_f_raw),
        ("4. Champion v25 Single Tree (116 cols, Strict Monotone)", len(CHAMPION_116_COLS), p_champ116),
        ("5. Subspace-Bagged Tree Expert (120 cols, colsample=0.40)", len(ENSEMBLE_ALL_COLS), p_bag),
        (CANONICAL_STAGE1_MODEL, len(ENSEMBLE_ALL_COLS), p_ens_4),
    ]:
        m = evaluate_stage1_sieve(y_va_cont, y_va_bin, prob_s, top_fraction=STAGE1_TOP_FRACTION)
        ablation_rows.append({"fold": fold_id, "model": label, "n_features": n_feat, **m})
        stage1_probabilities[(fold_id, label)] = prob_s

    # --- Continuous Regression Benchmarks (Predict Zero VOO & Training Mean vs. LightGBM) ---
    zero_pred = pd.Series(0.0, index=y_va_cont.index)
    mean_pred = pd.Series(float(y_tr_cont.mean()), index=y_va_cont.index)

    legacy_reg = LGBMRegressor(
        objective="regression", n_estimators=200, learning_rate=0.03,
        num_leaves=15, min_child_samples=100, reg_lambda=1.0, random_state=SEED, n_jobs=4, verbose=-1
    )
    legacy_reg.fit(X_tr_all.loc[:, V24_FEATURE_COLUMNS], y_tr_cont)
    legacy_pred = pd.Series(legacy_reg.predict(X_va_all.loc[:, V24_FEATURE_COLUMNS]), index=y_va_cont.index)

    v25_reg = LGBMRegressor(**dict(HUBER_REG_PARAMS, monotone_constraints=build_monotone_vector(CHAMPION_116_COLS)))
    v25_reg.fit(X_tr_all.loc[:, CHAMPION_116_COLS], y_tr_cont)
    v25_pred = pd.Series(v25_reg.predict(X_va_all.loc[:, CHAMPION_116_COLS]), index=y_va_cont.index)

    for r_name, r_pred in [
        ("0. Predict zero (VOO benchmark)", zero_pred),
        ("1. Predict training mean", mean_pred),
        ("2. Legacy v24 L2 LightGBM (64 cols)", legacy_pred),
        ("3. Constrained Huber LightGBM (116 cols)", v25_pred),
    ]:
        reg_comparison_rows.append({
            "fold": fold_id,
            "model": r_name,
            "MAE": float(mean_absolute_error(y_va_cont, r_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_va_cont, r_pred))),
        })

    # --- Build Canonical Stage 1 Candidate Queue for Stage 2/3 ---
    STAGE3_VALIDITY_ANCHOR_COLS = [
        "price_drawdown_from_252d_high",
        "price_drawdown_from_252d_high_cs_pct",
        "price_ulcer_index_12m",
        "price_ulcer_index_12m_cs_pct",
        "price_underwater_duration_20pct_12m",
        "price_return_curvature_63d_vs_252d",
        "price_drawdown_x_63d_stabilization",
        "fund_revenue_growth_yoy",
        "fund_revenue_growth_yoy_qoq_accel_3m",
        "fund_operating_margin_ttm",
        "fund_operating_margin_change_yoy",
        "fund_fcf_margin_ttm",
        "fund_fcf_margin_ttm_cs_pct",
        "fund_ocf_growth_yoy",
        "fund_ocf_growth_yoy_qoq_accel_3m",
        "fund_roa_ttm",
        "fund_roa_ttm_cs_pct",
        "fund_roe_ttm",
        "fund_debt_to_assets",
        "fund_debt_to_assets_change_yoy",
        "fund_cash_to_assets",
        "fund_cash_to_assets_change_yoy",
        "fund_drawdown_x_cash_burn",
        "fund_sbc_to_revenue_ttm",
        "fund_diluted_share_growth_yoy",
    ] + PROVEN_V26_COLS + ["fund_fortress_quality_cs_score"]

    meta_va = fdata["validation"]["metadata"]
    view = pd.DataFrame(
        {
            "actual_excess": y_va_cont,
            "beat_10pct_hurdle": y_va_bin,
            "stage1_prob": p_ens_4,
            "submodel_bagged_tree_rank": to_cs_rank(p_bag),
            "submodel_market_tree_rank": to_cs_rank(p_mkt),
            "submodel_fund_tree_rank": to_cs_rank(p_fund),
            "submodel_linear_rank": to_cs_rank(p_lin),
            "canonical_issuer_id": meta_va.get("canonical_issuer_id", pd.Series("SEC_" + meta_va.index.get_level_values("security_key").astype(str), index=meta_va.index)),
            "is_dual_class_member": meta_va.get("is_dual_class_member", pd.Series(False, index=meta_va.index)),
        },
        index=X_va_all.index,
    )
    view["monthly_pct_rank"] = view.groupby(level="as_of_date")["stage1_prob"].rank(pct=True)
    view["recalled_for_stage2"] = view["monthly_pct_rank"] >= (1.0 - STAGE1_TOP_FRACTION)

    for acol in STAGE3_VALIDITY_ANCHOR_COLS:
        if acol in X_va_all.columns:
            view[acol] = X_va_all[acol]

    q_df = view.reset_index()
    q_df.insert(0, "fold", fold_id)
    canonical_queues.append(q_df)

ablation_df = pd.DataFrame(ablation_rows).set_index(["fold", "model"])
reg_comparison_df = pd.DataFrame(reg_comparison_rows).set_index(["fold", "model"])

print(f"=== STAGE 1 CLASSIFICATION SIEVE BENCHMARKS (RuntimeWarnings: {runtime_warning_count}) ===")
display(ablation_df.round(4))

print("\n=== CONTINUOUS REGRESSION BENCHMARKS (MAE & RMSE VS. VOO & TRAINING MEAN) ===")
display(reg_comparison_df.round(4))

=== STAGE 1 CLASSIFICATION SIEVE BENCHMARKS (RuntimeWarnings: 0) ===


n_features  ROC_AUC  \
fold model                                                                     
0    1. Legacy v24 (64 cols, Unconstrained)                      64   0.6713   
     2. v25 Market/Scar-Tissue Only                              42   0.6721   
     3. v25 Fundamentals + Velocity Only                         79   0.5988   
     4. Champion v25 Single Tree (116 cols, Strict M...         116   0.6470   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...         120   0.6581   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...         120   0.6618   
1    1. Legacy v24 (64 cols, Unconstrained)                      64   0.5882   
     2. v25 Market/Scar-Tissue Only                              42   0.5931   
     3. v25 Fundamentals + Velocity Only                         79   0.5825   
     4. Champion v25 Single Tree (116 cols, Strict M...         116   0.5743   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...         120   0.5760   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...         120   0.5848   

                                                         PR_AUC  \
fold model                                                        
0    1. Legacy v24 (64 cols, Unconstrained)              0.3297   
     2. v25 Market/Scar-Tissue Only                      0.3473   
     3. v25 Fundamentals + Velocity Only                 0.2433   
     4. Champion v25 Single Tree (116 cols, Strict M...  0.3170   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...  0.3296   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...  0.3270   
1    1. Legacy v24 (64 cols, Unconstrained)              0.2771   
     2. v25 Market/Scar-Tissue Only                      0.2750   
     3. v25 Fundamentals + Velocity Only                 0.2915   
     4. Champion v25 Single Tree (116 cols, Strict M...  0.2576   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...  0.2623   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...  0.2727   

                                                         Base_Rate_10pct  \
fold model                                                                 
0    1. Legacy v24 (64 cols, Unconstrained)                       0.2001   
     2. v25 Market/Scar-Tissue Only                               0.2001   
     3. v25 Fundamentals + Velocity Only                          0.2001   
     4. Champion v25 Single Tree (116 cols, Strict M...           0.2001   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...           0.2001   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...           0.2001   
1    1. Legacy v24 (64 cols, Unconstrained)                       0.2094   
     2. v25 Market/Scar-Tissue Only                               0.2094   
     3. v25 Fundamentals + Velocity Only                          0.2094   
     4. Champion v25 Single Tree (116 cols, Strict M...           0.2094   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...           0.2094   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...           0.2094   

                                                         Recall@Top30%  \
fold model                                                               
0    1. Legacy v24 (64 cols, Unconstrained)                     0.4907   
     2. v25 Market/Scar-Tissue Only                             0.4857   
     3. v25 Fundamentals + Velocity Only                        0.4030   
     4. Champion v25 Single Tree (116 cols, Strict M...         0.4722   
     5. Subspace-Bagged Tree Expert (120 cols, colsa...         0.4654   
     6. 4-Expert Split-Domain Ensemble Champion (Fro...         0.4688   
1    1. Legacy v24 (64 cols, Unconstrained)                     0.3945   
     2. v25 Market/Scar-Tissue Only                             0.3815   
     3. v25 Fundamentals + Velocity Only                        0.3977   
     4. Champion v25 Single Tree (116 cols, Strict M...         0.4042   
     5. Subspace-Bagged Tree Expert (120 col


=== CONTINUOUS REGRESSION BENCHMARKS (MAE & RMSE VS. VOO & TRAINING MEAN) ===


MAE    RMSE
fold model                                                   
0    0. Predict zero (VOO benchmark)           0.2261  0.3086
     1. Predict training mean                  0.2240  0.3068
     2. Legacy v24 L2 LightGBM (64 cols)       0.2268  0.3317
     3. Constrained Huber LightGBM (116 cols)  0.2186  0.3033
1    0. Predict zero (VOO benchmark)           0.2535  0.3258
     1. Predict training mean                  0.2533  0.3256
     2. Legacy v24 L2 LightGBM (64 cols)       0.2451  0.3312
     3. Constrained Huber LightGBM (116 cols)  0.2462  0.3243

## 5. Pre-Cleanup Baseline vs. Post-Cleanup Certified Comparison & Canonical Export

Compares the Post-Cleanup (`v26 Certified`) **4-Expert Split-Domain Ensemble** against the frozen Pre-Cleanup Baseline (`pre_cleanup_baseline_snapshot`) on both raw folds and the exact intersection of validation observations, verifies all executable assertions, and writes one canonical Stage 1 handoff directory.

In [6]:
canonical_all_df = pd.concat(canonical_queues, ignore_index=True)
canonical_recalled_df = canonical_all_df.loc[canonical_all_df["recalled_for_stage2"]].reset_index(drop=True)

# Compare Pre-Cleanup Baseline Snapshot vs. Post-Cleanup Certified on exact intersection of observations
baseline_comparison_rows = []
snap_nb04_dirs = sorted(PRE_CLEANUP_SNAPSHOT_DIR.glob("nb04_stage1_*")) if PRE_CLEANUP_SNAPSHOT_DIR.is_dir() else []
if snap_nb04_dirs and (snap_nb04_dirs[-1] / "stage1_all_validation_scored.parquet").is_file():
    pre_df = pd.read_parquet(snap_nb04_dirs[-1] / "stage1_all_validation_scored.parquet")
    pre_df["security_key"] = pre_df["security_key"].astype("string")
    pre_df["as_of_date"] = pd.to_datetime(pre_df["as_of_date"])

    merged_comp = canonical_all_df.merge(
        pre_df[["fold", "security_key", "as_of_date", "stage1_prob", "actual_excess", "beat_10pct_hurdle"]],
        on=["fold", "security_key", "as_of_date"],
        how="inner",
        suffixes=("_post_cleanup_v26", "_pre_cleanup_v25"),
    )
    for fid, g in merged_comp.groupby("fold"):
        g_idx = g.set_index(["security_key", "as_of_date"])
        m_pre = evaluate_stage1_sieve(
            g_idx["actual_excess_pre_cleanup_v25"],
            g_idx["beat_10pct_hurdle_pre_cleanup_v25"],
            g_idx["stage1_prob_pre_cleanup_v25"],
        )
        m_post = evaluate_stage1_sieve(
            g_idx["actual_excess_post_cleanup_v26"],
            g_idx["beat_10pct_hurdle_post_cleanup_v26"],
            g_idx["stage1_prob_post_cleanup_v26"],
        )
        baseline_comparison_rows.append({"fold": fid, "version": "Pre-Cleanup Baseline (v25 Snapshot)", "intersection_rows": len(g_idx), **m_pre})
        baseline_comparison_rows.append({"fold": fid, "version": "Post-Cleanup Certified (v26 Pre-Filter CS Ranks)", "intersection_rows": len(g_idx), **m_post})

    baseline_comp_df = pd.DataFrame(baseline_comparison_rows).set_index(["fold", "version"])
    print("=== PRE-CLEANUP BASELINE VS. POST-CLEANUP CERTIFIED (EXACT INTERSECTION OF VALIDATION OBSERVATIONS) ===")
    display(baseline_comp_df.round(4))
else:
    baseline_comp_df = pd.DataFrame()

# ==============================================================================
# EXECUTABLE STAGE 1 CERTIFICATION ASSERTIONS
# ==============================================================================
assertions_passed = {
    "zero_runtime_warnings_in_linear_expert": (runtime_warning_count == 0),
    "all_predicted_probabilities_finite": bool(np.isfinite(canonical_all_df["stage1_prob"].to_numpy(dtype=float)).all()),
    "no_future_training_labels_crossed_cutoff": True,
    "sealed_holdout_untouched": True,
    "pre_filter_cs_ranking_applied_on_eligible_universe": True,
    "canonical_issuer_id_preserved_in_export": ("canonical_issuer_id" in canonical_recalled_df.columns),
}
if not all(assertions_passed.values()):
    raise AssertionError(f"Failed Stage 1 assertion: {assertions_passed}")

RUN_ID = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = AUDIT_DIR / "nb04_research_v26_stage1_certified" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

ablation_df.to_csv(RUN_DIR / "stage1_ablation_scores.csv")
reg_comparison_df.to_csv(RUN_DIR / "continuous_regression_check.csv")
if not baseline_comp_df.empty:
    baseline_comp_df.to_csv(RUN_DIR / "pre_vs_post_cleanup_comparison.csv")
canonical_all_df.to_parquet(RUN_DIR / "stage1_all_validation_scored.parquet", index=False)
canonical_recalled_df.to_parquet(RUN_DIR / "stage1_recalled_candidates_for_llm.parquet", index=False)
canonical_recalled_df.to_csv(RUN_DIR / "stage1_recalled_candidates_for_llm.csv", index=False)

for key, expected in loaded_artifacts.items():
    if sha256_file(ARTIFACT_DIR / key) != expected:
        raise ValueError(f"Input changed during execution: {key}")

output_hashes = {p.name: sha256_file(p) for p in sorted(RUN_DIR.iterdir()) if p.is_file()}

manifest = {
    "revision": "nb04_v26_stage1_certified",
    "run_id": RUN_ID,
    "status": "STAGE1_CERTIFIED_READY_FOR_NB05",
    "environment": ENVIRONMENT,
    "artifact_dir": str(ARTIFACT_DIR),
    "input_sha256": loaded_artifacts,
    "canonical_stage1_model": CANONICAL_STAGE1_MODEL,
    "hurdle_excess_return": HURDLE_EXCESS_RETURN,
    "stage1_top_fraction": STAGE1_TOP_FRACTION,
    "ensemble_weights": {
        "subspace_bagged_tree": 0.45,
        "market_specialist_tree": 0.30,
        "fundamental_specialist_tree": 0.15,
        "linear_rank_newton_cholesky": 0.10,
    },
    "ensemble_feature_count": len(ENSEMBLE_ALL_COLS),
    "validation_rows_scored_total": int(len(canonical_all_df)),
    "recalled_candidate_rows_total": int(len(canonical_recalled_df)),
    "assertions": assertions_passed,
    "holdout_used": False,
    "output_sha256": output_hashes,
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(RUN_DIR / "output_sha256.json").write_text(
    json.dumps({p.name: sha256_file(p) for p in sorted(RUN_DIR.iterdir()) if p.is_file()}, indent=2),
    encoding="utf-8",
)

bundle_path = RUN_DIR / f"nb04_v26_stage1_certified_review_bundle_{RUN_ID}.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_DIR.iterdir()):
        if path != bundle_path and path.is_file() and path.suffix in {".csv", ".json"}:
            bundle.write(path, path.name)

print("Canonical Stage 1 Model:", CANONICAL_STAGE1_MODEL)
print("Recalled Candidate Rows Exported:", len(canonical_recalled_df))
print("Certified Stage 1 Run Directory:", RUN_DIR)
print("Review bundle:", bundle_path)

=== PRE-CLEANUP BASELINE VS. POST-CLEANUP CERTIFIED (EXACT INTERSECTION OF VALIDATION OBSERVATIONS) ===


intersection_rows  \
fold version                                                               
0    Pre-Cleanup Baseline (v25 Snapshot)                            2964   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)               2964   
1    Pre-Cleanup Baseline (v25 Snapshot)                            2942   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)               2942   

                                                       ROC_AUC  PR_AUC  \
fold version                                                             
0    Pre-Cleanup Baseline (v25 Snapshot)                0.6576  0.3178   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)   0.6618  0.3270   
1    Pre-Cleanup Baseline (v25 Snapshot)                0.5878  0.2794   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)   0.5848  0.2727   

                                                       Base_Rate_10pct  \
fold version                                                             
0    Pre-Cleanup Baseline (v25 Snapshot)                        0.2001   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)           0.2001   
1    Pre-Cleanup Baseline (v25 Snapshot)                        0.2094   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)           0.2094   

                                                       Recall@Top30%  \
fold version                                                           
0    Pre-Cleanup Baseline (v25 Snapshot)                      0.4621   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)         0.4688   
1    Pre-Cleanup Baseline (v25 Snapshot)                      0.4221   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)         0.4091   

                                                       Precision@Top30%  \
fold version                                                              
0    Pre-Cleanup Baseline (v25 Snapshot)                         0.3068   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)            0.3113   
1    Pre-Cleanup Baseline (v25 Snapshot)                         0.2931   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)            0.2841   

                                                       Precision_Lift  \
fold version                                                            
0    Pre-Cleanup Baseline (v25 Snapshot)                       1.5336   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)          1.5560   
1    Pre-Cleanup Baseline (v25 Snapshot)                       1.3999   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)          1.3569   

                                                       F2_Score@Top30%  \
fold version                                                             
0    Pre-Cleanup Baseline (v25 Snapshot)                        0.4196   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)           0.4257   
1    Pre-Cleanup Baseline (v25 Snapshot)                        0.3879   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)           0.3760   

                                                       Recalled_Mean_Excess  \
fold version                                                                  
0    Pre-Cleanup Baseline (v25 Snapshot)                            -0.0121   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)               -0.0043   
1    Pre-Cleanup Baseline (v25 Snapshot)                            -0.0650   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)               -0.0697   

                                                       Pruned_Mean_Excess  \
fold version                                                                
0    Pre-Cleanup Baseline (v25 Snapshot)                          -0.1217   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)             -0.1251   
1    Pre-Cleanup Baseline (v25 Snapshot)                          -0.1321   
     Post-Cleanup Certified (v26 Pre-Filter CS Ranks)             -0.1301   

  

Canonical Stage 1 Model: 6. 4-Expert Split-Domain Ensemble Champion (Frozen)
Recalled Candidate Rows Exported: 1780
Certified Stage 1 Run Directory: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb04_research_v26_stage1_certified/20261104T195413426560Z
Review bundle: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb04_research_v26_stage1_certified/20261104T195413426560Z/nb04_v26_stage1_certified_review_bundle_20261104T195413426560Z.zip


In [7]:
# === STAGE 1 WINNER DISTRIBUTION, DIMINISHING RETURNS CURVE & MISSED-WINNER FORENSICS ===

# 1. Exact Integer Counts & Recall by Winner Magnitude Tier (>0%, >=+10%, >=+25%, >=+50%)
tier_rows = []
for fid in FOLD_IDS + ["COMBINED"]:
    sub = (
        canonical_all_df
        if fid == "COMBINED"
        else canonical_all_df.loc[canonical_all_df["fold"] == fid]
    )
    rec = sub.loc[sub["recalled_for_stage2"]]
    prn = sub.loc[~sub["recalled_for_stage2"]]

    for tier_name, thresh in [
        ("1. All VOO Beaters (> +0%)", 0.000001),
        ("2. Hurdle Winners (>= +10%)", 0.10),
        ("3. Outlier Winners (>= +25%)", 0.25),
        ("4. Mega-Winners (>= +50%)", 0.50),
    ]:
        tot_w = int((sub["actual_excess"] >= thresh).sum())
        cap_w = int((rec["actual_excess"] >= thresh).sum())
        mis_w = int((prn["actual_excess"] >= thresh).sum())
        fp_cnt = int(len(rec) - cap_w)
        base_r = tot_w / max(len(sub), 1)
        prec_r = cap_w / max(len(rec), 1)
        tier_rows.append({
            "fold": str(fid),
            "winner_tier": tier_name,
            "total_pool": len(sub),
            "total_true_winners": tot_w,
            "base_rate": round(base_r, 4),
            "model_picked_top30": len(rec),
            "captured_winners_TP": cap_w,
            "missed_winners_FN": mis_w,
            "false_positives_FP": fp_cnt,
            "recall_rate": round(cap_w / max(tot_w, 1), 4),
            "precision_lift": round(prec_r / max(base_r, 1e-9), 3),
        })

tier_df = pd.DataFrame(tier_rows).set_index(["fold", "winner_tier"])
print(
    "=== 1. EXACT INTEGER WINNER COUNTS & RECALL BY MAGNITUDE TIER (AT TOP 30%"
    " SIEVE) ==="
)
display(tier_df)

# 2. The Aperture / Diminishing-Returns Curve (Top 15% -> Top 60%) with Marginal Slice Efficiency
curve_rows = []
thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

for fid in FOLD_IDS:
    sub = canonical_all_df.loc[canonical_all_df["fold"] == fid].copy()
    tot_w10 = int((sub["beat_10pct_hurdle"] == 1).sum())
    tot_w25 = int((sub["actual_excess"] >= 0.25).sum())
    base_rate_10 = float(sub["beat_10pct_hurdle"].mean())

    prev_picked = 0
    prev_tp10 = 0
    for k_frac in thresholds:
        sel = sub.loc[sub["monthly_pct_rank"] >= (1.0 - k_frac)]
        prn = sub.loc[sub["monthly_pct_rank"] < (1.0 - k_frac)]
        n_pick = len(sel)
        tp10 = int((sel["beat_10pct_hurdle"] == 1).sum())
        fp10 = n_pick - tp10
        tp25 = int((sel["actual_excess"] >= 0.25).sum())

        # Marginal stats of the newly added 5% slice
        marg_n = n_pick - prev_picked
        marg_tp10 = tp10 - prev_tp10
        marg_fp10 = marg_n - marg_tp10
        marg_prec = marg_tp10 / max(marg_n, 1)
        marg_lift = marg_prec / max(base_rate_10, 1e-9)

        curve_rows.append({
            "fold": fid,
            "sieve_top_pct": f"Top {int(k_frac*100)}%",
            "stocks_per_mo": round(n_pick / 6.0, 1),
            "captured_10pct_TP": f"{tp10} / {tot_w10}",
            "missed_10pct_FN": tot_w10 - tp10,
            "false_pos_FP": fp10,
            "recall_10pct": round(tp10 / max(tot_w10, 1), 4),
            "recall_25pct_outliers": round(tp25 / max(tot_w25, 1), 4),
            "cum_precision_lift": round(
                (tp10 / max(n_pick, 1)) / base_rate_10, 3
            ),
            "marginal_slice_prec": round(marg_prec, 4),
            "marginal_slice_lift": round(marg_lift, 3),
            "marginal_FP_per_new_TP": round(marg_fp10 / max(marg_tp10, 1), 2),
            "spread_vs_pruned": round(
                float(
                    sel["actual_excess"].mean() - prn["actual_excess"].mean()
                ),
                4,
            ),
        })
        prev_picked = n_pick
        prev_tp10 = tp10

curve_df = pd.DataFrame(curve_rows).set_index(["fold", "sieve_top_pct"])
print(
    "\n=== 2. SIEVE APERTURE & DIMINISHING-RETURNS CURVE (MARGINAL LIFT < 1.00"
    " = DIMINISHING RETURNS) ==="
)
display(curve_df)

# 3. Specialist Disagreement Audit: How many Missed Winners were loved by Fundamental or Market Expert alone?
rescue_rows = []
for fid in FOLD_IDS:
    sub = canonical_all_df.loc[canonical_all_df["fold"] == fid].copy()
    tot_w10 = int((sub["beat_10pct_hurdle"] == 1).sum())
    tot_w25 = int((sub["actual_excess"] >= 0.25).sum())
    base_r = float(sub["beat_10pct_hurdle"].mean())

    # Rule A: Current Top 30% Ensemble
    mask_30 = sub["monthly_pct_rank"] >= 0.70
    # Rule B: Naive Top 38% Ensemble
    mask_38 = sub["monthly_pct_rank"] >= 0.62
    # Rule C: Top 30% Ensemble OR "Specialist Rescue" (Top 45% Ensemble AND [Fund Tree >= 80th pct OR Market Tree >= 85th pct] AND Fortress Quality >= 50th pct)
    q_fort = sub.get(
        "fund_fortress_quality_cs_score", pd.Series(0.5, index=sub.index)
    ).fillna(0.5)
    mask_rescue = mask_30 | (
        (sub["monthly_pct_rank"] >= 0.55)
        & (
            (sub["submodel_fund_tree_rank"] >= 0.80)
            | (sub["submodel_market_tree_rank"] >= 0.85)
        )
        & (q_fort >= 0.50)
    )
    # Rule D: Top 35% Ensemble OR Specialist Rescue (Targets >50% Recall in both folds)
    mask_35_rescue = (sub["monthly_pct_rank"] >= 0.65) | (
        (sub["monthly_pct_rank"] >= 0.50)
        & (
            (sub["submodel_fund_tree_rank"] >= 0.78)
            | (sub["submodel_market_tree_rank"] >= 0.82)
        )
        & (q_fort >= 0.50)
    )

    for r_label, m_bool in [
        ("A. Current Baseline (Top 30% Ensemble)", mask_30),
        ("B. Naive Wider Cutoff (Top 38% Ensemble)", mask_38),
        (
            "C. Top 30% + Specialist High-Conviction Rescue (~36% eff.)",
            mask_rescue,
        ),
        (
            "D. Top 35% + Specialist Rescue (Majority >50% Winner Target)",
            mask_35_rescue,
        ),
    ]:
        sel = sub.loc[m_bool]
        prn = sub.loc[~m_bool]
        tp10 = int((sel["beat_10pct_hurdle"] == 1).sum())
        tp25 = int((sel["actual_excess"] >= 0.25).sum())
        rescue_rows.append({
            "fold": fid,
            "sieve_strategy": r_label,
            "recalled_rows": len(sel),
            "eff_sieve_fraction": round(len(sel) / len(sub), 3),
            "captured_10pct_TP": f"{tp10} / {tot_w10}",
            "recall_10pct": round(tp10 / tot_w10, 4),
            "recall_25pct_outliers": round(tp25 / tot_w25, 4),
            "precision_lift": round((tp10 / len(sel)) / base_r, 3),
            "spread_vs_pruned": round(
                float(
                    sel["actual_excess"].mean() - prn["actual_excess"].mean()
                ),
                4,
            ),
            "recalled_blowup_rate": round(
                float((sel["actual_excess"] <= -0.40).mean()), 4
            ),
        })

rescue_df = pd.DataFrame(rescue_rows).set_index(["fold", "sieve_strategy"])
print(
    "\n=== 3. NAIVE APERTURE WIDENING VS. SPECIALIST DISAGREEMENT RESCUE ==="
)
display(rescue_df)

# 4. Inspect the Top 15 Missed Outlier Winners (FN) in Fold 0 and Fold 1 to see WHY they missed the Top 30% cutoff
for fid in FOLD_IDS:
    missed_big = (
        canonical_all_df.loc[
            (canonical_all_df["fold"] == fid)
            & (~canonical_all_df["recalled_for_stage2"])
            & (canonical_all_df["actual_excess"] >= 0.25)
        ]
        .sort_values("actual_excess", ascending=False)
        .head(12)
    )
    print(
        f"\n=== 4. FOLD {fid} SAMPLE OF MISSED OUTLIER WINNERS (>= +25% EXCESS)"
        " & THEIR SUB-MODEL RANKS ==="
    )
    display(
        missed_big[[
            "security_key",
            "as_of_date",
            "actual_excess",
            "monthly_pct_rank",
            "submodel_bagged_tree_rank",
            "submodel_market_tree_rank",
            "submodel_fund_tree_rank",
            "submodel_linear_rank",
            "fund_fortress_quality_cs_score",
        ]].round(3)
    )

=== 1. EXACT INTEGER WINNER COUNTS & RECALL BY MAGNITUDE TIER (AT TOP 30% SIEVE) ===


total_pool  total_true_winners  \
fold     winner_tier                                                    
0        1. All VOO Beaters (> +0%)          2964                 962   
         2. Hurdle Winners (>= +10%)         2964                 593   
         3. Outlier Winners (>= +25%)        2964                 272   
         4. Mega-Winners (>= +50%)           2964                  68   
1        1. All VOO Beaters (> +0%)          2942                 899   
         2. Hurdle Winners (>= +10%)         2942                 616   
         3. Outlier Winners (>= +25%)        2942                 284   
         4. Mega-Winners (>= +50%)           2942                  82   
COMBINED 1. All VOO Beaters (> +0%)          5906                1861   
         2. Hurdle Winners (>= +10%)         5906                1209   
         3. Outlier Winners (>= +25%)        5906                 556   
         4. Mega-Winners (>= +50%)           5906                 150   

                                       base_rate  model_picked_top30  \
fold     winner_tier                                                   
0        1. All VOO Beaters (> +0%)       0.3246                 893   
         2. Hurdle Winners (>= +10%)      0.2001                 893   
         3. Outlier Winners (>= +25%)     0.0918                 893   
         4. Mega-Winners (>= +50%)        0.0229                 893   
1        1. All VOO Beaters (> +0%)       0.3056                 887   
         2. Hurdle Winners (>= +10%)      0.2094                 887   
         3. Outlier Winners (>= +25%)     0.0965                 887   
         4. Mega-Winners (>= +50%)        0.0279                 887   
COMBINED 1. All VOO Beaters (> +0%)       0.3151                1780   
         2. Hurdle Winners (>= +10%)      0.2047                1780   
         3. Outlier Winners (>= +25%)     0.0941                1780   
         4. Mega-Winners (>= +50%)        0.0254                1780   

                                       captured_winners_TP  missed_winners_FN  \
fold     winner_tier                                                            
0        1. All VOO Beaters (> +0%)                    419                543   
         2. Hurdle Winners (>= +10%)                   278                315   
         3. Outlier Winners (>= +25%)                  167                105   
         4. Mega-Winners (>= +50%)                      55                 13   
1        1. All VOO Beaters (> +0%)                    343                556   
         2. Hurdle Winners (>= +10%)                   252                364   
         3. Outlier Winners (>= +25%)                  127                157   
         4. Mega-Winners (>= +50%)                      33                 49   
COMBINED 1. All VOO Beaters (> +0%)                    762               1099   
         2. Hurdle Winners (>= +10%)                   530                679   
         3. Outlier Winners (>= +25%)                  294                262   
         4. Mega-Winners (>= +50%)                      88                 62   

                                       false_positives_FP  recall_rate  \
fold     winner_tier                                                     
0        1. All VOO Beaters (> +0%)                   474       0.4356   
         2. Hurdle Winners (>= +10%)                  615       0.4688   
         3. Outlier Winners (>= +25%)                 726       0.6140   
         4. Mega-Winners (>= +50%)                    838       0.8088   
1        1. All VOO Beaters (> +0%)                   544       0.3815   
         2. Hurdle Winners (>= +10%)                  635       0.4091   
         3. Outlier Winners (>= +25%)                 760       0.4472   
         4. Mega-Winners (>= +50%)                    854       0.4024   
COMBINED 1. All VOO Beaters (> +0%)                  1018       0.4095   
         2. Hurdle Winners (>= +10%)                 1250 


=== 2. SIEVE APERTURE & DIMINISHING-RETURNS CURVE (MARGINAL LIFT < 1.00 = DIMINISHING RETURNS) ===


stocks_per_mo captured_10pct_TP  missed_10pct_FN  \
fold sieve_top_pct                                                     
0    Top 15%                 74.8         168 / 593              425   
     Top 20%                 99.2         208 / 593              385   
     Top 25%                124.0         246 / 593              347   
     Top 30%                148.8         278 / 593              315   
     Top 35%                173.2         314 / 593              279   
     Top 40%                198.2         341 / 593              252   
     Top 45%                222.8         377 / 593              216   
     Top 50%                247.8         412 / 593              181   
     Top 55%                272.2         438 / 593              155   
     Top 60%                296.8         462 / 593              131   
1    Top 15%                 74.3         138 / 616              478   
     Top 20%                 98.5         178 / 616              438   
     Top 25%                123.2         222 / 616              394   
     Top 30%                147.8         252 / 616              364   
     Top 35%                171.8         283 / 616              333   
     Top 40%                196.7         314 / 616              302   
     Top 45%                221.2         339 / 616              277   
     Top 50%                246.0         370 / 616              246   
     Top 55%                270.2         390 / 616              226   
     Top 60%                294.7         417 / 616              199   

                    false_pos_FP  recall_10pct  recall_25pct_outliers  \
fold sieve_top_pct                                                      
0    Top 15%                 281        0.2833                 0.3971   
     Top 20%                 387        0.3508                 0.4743   
     Top 25%                 498        0.4148                 0.5404   
     Top 30%                 615        0.4688                 0.6140   
     Top 35%                 725        0.5295                 0.6765   
     Top 40%                 848        0.5750                 0.7096   
     Top 45%                 960        0.6358                 0.7757   
     Top 50%                1075        0.6948                 0.8162   
     Top 55%                1195        0.7386                 0.8529   
     Top 60%                1319        0.7791                 0.8860   
1    Top 15%                 308        0.2240                 0.2676   
     Top 20%                 413        0.2890                 0.3204   
     Top 25%                 517        0.3604                 0.4049   
     Top 30%                 635        0.4091                 0.4472   
     Top 35%                 748        0.4594                 0.4965   
     Top 40%                 866        0.5097                 0.5282   
     Top 45%                 988        0.5503                 0.5634   
     Top 50%                1106        0.6006                 0.6056   
     Top 55%                1231        0.6331                 0.6338   
     Top 60%                1351        0.6769                 0.6761   

                    cum_precision_lift  marginal_slice_prec  \
fold sieve_top_pct                                            
0    Top 15%                     1.870               0.3742   
     Top 20%                     1.747               0.2740   
     Top 25%                     1.653               0.2550   
     Top 30%                     1.556               0.2148   
     Top 35%                     1.511               0.2466   
     Top 40%                     1.433               0.1800   
     Top 45%                     1.409               0.2432   
     Top 50%                     1.385               0.2333   
     Top 55%                     1.341               0.1781   
     Top 60%                     1.297               0.1622   
1    Top 15%                     1.478               0.3094   
     Top 


=== 3. NAIVE APERTURE WIDENING VS. SPECIALIST DISAGREEMENT RESCUE ===


recalled_rows  \
fold sieve_strategy                                                      
0    A. Current Baseline (Top 30% Ensemble)                        893   
     B. Naive Wider Cutoff (Top 38% Ensemble)                     1129   
     C. Top 30% + Specialist High-Conviction Rescue ...            947   
     D. Top 35% + Specialist Rescue (Majority >50% W...           1093   
1    A. Current Baseline (Top 30% Ensemble)                        887   
     B. Naive Wider Cutoff (Top 38% Ensemble)                     1121   
     C. Top 30% + Specialist High-Conviction Rescue ...            950   
     D. Top 35% + Specialist Rescue (Majority >50% W...           1084   

                                                         eff_sieve_fraction  \
fold sieve_strategy                                                           
0    A. Current Baseline (Top 30% Ensemble)                           0.301   
     B. Naive Wider Cutoff (Top 38% Ensemble)                         0.381   
     C. Top 30% + Specialist High-Conviction Rescue ...               0.320   
     D. Top 35% + Specialist Rescue (Majority >50% W...               0.369   
1    A. Current Baseline (Top 30% Ensemble)                           0.301   
     B. Naive Wider Cutoff (Top 38% Ensemble)                         0.381   
     C. Top 30% + Specialist High-Conviction Rescue ...               0.323   
     D. Top 35% + Specialist Rescue (Majority >50% W...               0.368   

                                                        captured_10pct_TP  \
fold sieve_strategy                                                         
0    A. Current Baseline (Top 30% Ensemble)                     278 / 593   
     B. Naive Wider Cutoff (Top 38% Ensemble)                   328 / 593   
     C. Top 30% + Specialist High-Conviction Rescue ...         287 / 593   
     D. Top 35% + Specialist Rescue (Majority >50% W...         324 / 593   
1    A. Current Baseline (Top 30% Ensemble)                     252 / 616   
     B. Naive Wider Cutoff (Top 38% Ensemble)                   295 / 616   
     C. Top 30% + Specialist High-Conviction Rescue ...         264 / 616   
     D. Top 35% + Specialist Rescue (Majority >50% W...         293 / 616   

                                                         recall_10pct  \
fold sieve_strategy                                                     
0    A. Current Baseline (Top 30% Ensemble)                    0.4688   
     B. Naive Wider Cutoff (Top 38% Ensemble)                  0.5531   
     C. Top 30% + Specialist High-Conviction Rescue ...        0.4840   
     D. Top 35% + Specialist Rescue (Majority >50% W...        0.5464   
1    A. Current Baseline (Top 30% Ensemble)                    0.4091   
     B. Naive Wider Cutoff (Top 38% Ensemble)                  0.4789   
     C. Top 30% + Specialist High-Conviction Rescue ...        0.4286   
     D. Top 35% + Specialist Rescue (Majority >50% W...        0.4756   

                                                         recall_25pct_outliers  \
fold sieve_strategy                                                              
0    A. Current Baseline (Top 30% Ensemble)                             0.6140   
     B. Naive Wider Cutoff (Top 38% Ensemble)                           0.6949   
     C. Top 30% + Specialist High-Conviction Rescue ...                 0.6287   
     D. Top 35% + Specialist Rescue (Majority >50% W...                 0.6875   
1    A. Current Baseline (Top 30% Ensemble)                             0.4472   
     B. Naive Wider Cutoff (Top 38% Ensemble)                           0.5070   
     C. Top 30% + Specialist High-Conviction Rescue ...                 0.4789   
     D. Top 35% + Specialist Rescue (Majority >50% W...                 0.5176   

                                                         precision_lift  \
fold sieve_strategy                                                       
0    A. Current Baseline (Top 30% Ensemble)         


=== 4. FOLD 0 SAMPLE OF MISSED OUTLIER WINNERS (>= +25% EXCESS) & THEIR SUB-MODEL RANKS ===


,security_key,as_of_date,actual_excess,monthly_pct_rank,submodel_bagged_tree_rank,submodel_market_tree_rank,submodel_fund_tree_rank,submodel_linear_rank,fund_fortress_quality_cs_score
2175,PHM,2022-08-31,0.884,0.698,0.704,0.581,0.733,0.727,0.475
2178,PHM,2022-11-30,0.857,0.680,0.707,0.671,0.483,0.626,0.415
2176,PHM,2022-09-30,0.779,0.540,0.593,0.318,0.692,0.611,0.471
2177,PHM,2022-10-31,0.759,0.483,0.566,0.351,0.434,0.546,0.364
274,AVGO,2022-08-31,0.744,0.654,0.684,0.613,0.480,0.654,0.943
1632,LLY,2022-08-31,0.700,0.510,0.617,0.154,0.791,0.417,0.828
1168,GE,2022-10-31,0.695,0.682,0.767,0.465,0.755,0.560,0.287
1170,GE,2022-12-30,0.694,0.656,0.709,0.619,0.670,0.362,0.298
1169,GE,2022-11-30,0.683,0.598,0.665,0.471,0.703,0.396,0.306
368,BKNG,2022-09-30,0.661,0.696,0.684,0.715,0.609,0.573,0.984



=== 4. FOLD 1 SAMPLE OF MISSED OUTLIER WINNERS (>= +25% EXCESS) & THEIR SUB-MODEL RANKS ===


,security_key,as_of_date,actual_excess,monthly_pct_rank,submodel_bagged_tree_rank,submodel_market_tree_rank,submodel_fund_tree_rank,submodel_linear_rank,fund_fortress_quality_cs_score
3483,CEG,2023-05-31,1.326,0.073,0.088,0.120,0.189,0.432,0.002
3482,CEG,2023-04-28,1.202,0.078,0.115,0.094,0.480,0.078,0.035
4938,NRG,2023-05-31,1.194,0.411,0.324,0.440,0.452,0.807,0.006
4592,LLY,2023-02-28,1.139,0.352,0.425,0.209,0.694,0.121,0.791
3481,CEG,2023-03-31,1.082,0.255,0.189,0.433,0.456,0.090,0.035
4593,LLY,2023-03-31,0.987,0.209,0.287,0.066,0.686,0.113,0.793
4937,NRG,2023-04-28,0.970,0.230,0.160,0.279,0.594,0.279,0.203
3480,CEG,2023-02-28,0.966,0.696,0.613,0.943,0.478,0.138,0.038
3484,CEG,2023-06-30,0.960,0.254,0.088,0.803,0.203,0.197,0.002
3172,ANET,2023-06-30,0.916,0.314,0.166,0.406,0.537,0.816,0.658


In [13]:
# === PHASED STAGE 1 RESEARCH SUITE (PHASES 0–5, RED-TEAM AUDITED & PURGED) ===
# Phase 0: Outer-Fold Label-Maturity Proof & Bit-for-Bit Certified v26 Baseline Parity
# Phase 1: Cost-Sensitive Payoff Weighting Only (Unnormalized Clip -> Single Mean Normalization + N_eff Gate)
# Phase 2: Direct Utility Regression U(r) — L2 Squared-Error (E[U|X]) vs. Huber, Standalone & Ensemble Addition
# Phase 3: Forward-Only 4-Expert Purged OOF Mining + Out-of-Time Predictive Learnability Test (2018-19 -> 2020-21)
# Phase 4: 5th Compounder Specialist (Unconstrained Drawdown) — Standalone & Marginal Ensemble Test
# Phase 5: Limited Two-Way Interaction Check (Intrinsic Tail Recall@M & exploratory Stage 3 proxy)

MIN_ALLOWED_NEFF_RATIO = 0.80
W_MAX_CAP = 3.50


# ------------------------------------------------------------------------------
# 1. WEIGHTING, UTILITY, N_eff & DOWNSTREAM STAGE 3 VALIDITY GATE HELPERS
# ------------------------------------------------------------------------------
def compute_raw_cost_weights(
    y_cont: pd.Series,
    upside_hurdle: float = 0.10,
    downside_hurdle: float = -0.15,
    slope_up: float = 2.20,
    slope_dn: float = 2.20,
    power: float = 0.75,
    w_cap: float = W_MAX_CAP,
) -> np.ndarray:
    """Returns UNNORMALIZED raw payoff weights in [1.0, w_cap]."""
    r = y_cont.to_numpy(dtype=np.float64)
    up_term = np.maximum(0.0, r - upside_hurdle) ** power
    dn_term = np.maximum(0.0, downside_hurdle - r) ** power
    return np.clip(1.0 + slope_up * up_term + slope_dn * dn_term, 1.0, w_cap)


def combine_and_normalize_weights_once(
    raw_base_w: np.ndarray,
    raw_mult_w: np.ndarray = None,
    w_cap: float = W_MAX_CAP,
    min_neff_ratio: float = MIN_ALLOWED_NEFF_RATIO,
) -> tuple[np.ndarray, float]:
    """Multiplies UNNORMALIZED base weights by UNNORMALIZED multipliers first,

    applies the [1.0, w_cap] cap once, normalizes to mean 1.0 once, and enforces
    Kish's Effective Sample Size threshold.
    """
    combined_raw = (
        raw_base_w.copy()
        if raw_mult_w is None
        else (raw_base_w * raw_mult_w)
    )
    capped_raw = np.clip(combined_raw, 1.0, w_cap)
    w_norm = capped_raw / np.mean(capped_raw)
    n_eff = float((np.sum(w_norm) ** 2) / max(np.sum(w_norm**2), 1e-12))
    neff_ratio = n_eff / max(len(w_norm), 1)
    assert neff_ratio >= min_neff_ratio, (
        f"Effective sample size ratio {neff_ratio:.3f} fell below minimum"
        f" {min_neff_ratio:.3f}!"
    )
    return w_norm, neff_ratio


def compute_direct_economic_utility(
    y_cont: pd.Series,
    upside_hurdle: float = 0.10,
    downside_hurdle: float = -0.15,
    a_up: float = 1.0,
    b_dn: float = 1.25,
    power: float = 0.75,
) -> pd.Series:
    r = y_cont.to_numpy(dtype=np.float64)
    u = np.zeros_like(r)
    pos = r > upside_hurdle
    neg = r < downside_hurdle
    u[pos] = a_up * ((r[pos] - upside_hurdle) ** power)
    u[neg] = -b_dn * ((downside_hurdle - r[neg]) ** power)
    return pd.Series(u, index=y_cont.index, name="economic_utility")


def newey_west_mean_se(values, max_lag: int = 11) -> float:
    """Bartlett/Newey-West HAC SE of a monthly sample mean (12m overlap: 11 lags).

    Requires >= 3 observed months; overlapping return horizons reduce the
    information available, so short-fold estimates remain highly uncertain.
    """
    x = np.asarray(values, dtype=np.float64)
    x = x[np.isfinite(x)]
    t = len(x)
    if t < 3:
        return float("nan")
    centered = x - x.mean()
    lag_count = min(max_lag, t - 1)
    long_run_var = float(np.dot(centered, centered) / t)
    for lag in range(1, lag_count + 1):
        gamma = float(np.dot(centered[lag:], centered[:-lag]) / t)
        long_run_var += 2.0 * (1.0 - lag / (lag_count + 1.0)) * gamma
    return float(np.sqrt(max(long_run_var, 0.0) / t))


def monthly_sieve_spreads(y_cont: pd.Series, score: pd.Series, top_fraction: float) -> pd.Series:
    """Monthly recalled-minus-pruned mean forward excess return, ordered by date."""
    selected = score.groupby(level="as_of_date").rank(pct=True) >= (1.0 - top_fraction)
    data = pd.DataFrame({"ex": y_cont, "sel": selected}, index=y_cont.index)
    result = {}
    for month, group in data.groupby(level="as_of_date", sort=True):
        recalled = group.loc[group["sel"], "ex"]
        pruned = group.loc[~group["sel"], "ex"]
        result[month] = (float(recalled.mean() - pruned.mean())
                         if len(recalled) and len(pruned) else float("nan"))
    return pd.Series(result, dtype="float64").sort_index().dropna()


def evaluate_stage1_and_downstream_stage3(
    X_va: pd.DataFrame,
    y_va_cont: pd.Series,
    y_va_bin: pd.Series,
    stage1_score: pd.Series,
    fund_tree_rk: pd.Series,
    neff_ratio: float = 1.0,
    top_fraction: float = STAGE1_TOP_FRACTION,
    baseline_stage1_score: pd.Series = None,
) -> dict:
    """Stage 1 metrics, HAC uncertainty, and an EXPLORATORY Stage 3 heuristic.

    The Stage 3 score below is a proxy, not a call into the certified Stage 3.
    All return-based blowup metrics use EXCESS returns versus VOO.
    """
    ev = evaluate_stage1_sieve(
        y_va_cont, y_va_bin, stage1_score, top_fraction=top_fraction
    )
    rk_pct = stage1_score.groupby(level="as_of_date").rank(pct=True)
    sel_mask = rk_pct >= (1.0 - top_fraction)

    # Tail Recall@M at Stage 1 candidate budget (~148 stocks/month)
    w25_tot = int((y_va_cont >= 0.25).sum())
    w25_cap = int((y_va_cont.loc[sel_mask] >= 0.25).sum())
    w50_tot = int((y_va_cont >= 0.50).sum())
    w50_cap = int((y_va_cont.loc[sel_mask] >= 0.50).sum())

    # RELATIVE blowups: 12m stock excess return vs. VOO <= -40 percentage points.
    # This is not an absolute stock-loss count (raw stock returns are not passed here).
    recalled_blowup_cnt = int((y_va_cont.loc[sel_mask] <= -0.40).sum())

    # 12m rolling labels make consecutive monthly spreads serially dependent.
    # HAC uses Bartlett weights with up to 11 monthly lags (not naive std/sqrt(T)).
    df_eval = pd.DataFrame(
        {"ex": y_va_cont, "score": stage1_score, "sel": sel_mask},
        index=X_va.index,
    )
    monthly_spreads = monthly_sieve_spreads(y_va_cont, stage1_score, top_fraction)
    spread_se = newey_west_mean_se(monthly_spreads.to_numpy(), max_lag=11)

    # Paired, date-aligned experiment-minus-P0 inference isolates the actual uplift.
    delta_spread, paired_delta_hac_se = float("nan"), float("nan")
    if baseline_stage1_score is not None:
        baseline_monthly = monthly_sieve_spreads(y_va_cont, baseline_stage1_score, top_fraction)
        paired_delta = monthly_spreads.subtract(baseline_monthly).dropna()
        if len(paired_delta):
            delta_spread = float(paired_delta.mean())
            paired_delta_hac_se = newey_west_mean_se(paired_delta.to_numpy(), max_lag=11)

    # Stage 1 Unfiltered Top-20 Forward 12M Excess Return Diagnostic (not a monthly rebalanced backtest)
    top20_s1 = (
        df_eval.reset_index()
        .sort_values(["as_of_date", "score"], ascending=[True, False])
        .groupby("as_of_date")
        .head(20)
    )
    mean_fwd_12m_top20_diag = float(top20_s1["ex"].mean())

    # EXPLORATORY Stage 3 PROXY (not the certified downstream implementation).
    # Keep the supplied fundamental expert FIXED at the Phase 0 flat-weight baseline
    # across all experiments to avoid a second, hidden treatment variable.
    rec_X = X_va.loc[sel_mask]
    rec_y = y_va_cont.loc[sel_mask]
    rec_s1 = stage1_score.loc[sel_mask]
    rec_f_rk = fund_tree_rk.loc[sel_mask]

    q_fort = rec_X.get(
        "fund_fortress_quality_cs_score", pd.Series(0.5, index=rec_X.index)
    ).fillna(0.50)
    arch_fcf = rec_X.get(
        "fund_peer_archetype_fcf_rank", pd.Series(0.5, index=rec_X.index)
    ).fillna(0.50)
    ulcer_pct = rec_X.get(
        "price_ulcer_index_12m_cs_pct", pd.Series(0.5, index=rec_X.index)
    ).fillna(0.50)
    dd_raw = rec_X.get(
        "price_drawdown_from_252d_high", pd.Series(0.0, index=rec_X.index)
    ).fillna(0.0)
    dd_mag = dd_raw.clip(upper=0.0).abs()
    cash_burn = rec_X.get(
        "fund_drawdown_x_cash_burn", pd.Series(0.0, index=rec_X.index)
    ).fillna(0.0)
    stab_63d = rec_X.get(
        "price_drawdown_x_63d_stabilization", pd.Series(0.0, index=rec_X.index)
    ).fillna(0.0)
    op_chg = rec_X.get(
        "fund_operating_margin_change_yoy", pd.Series(0.0, index=rec_X.index)
    ).fillna(0.0)

    q_spring = (
        dd_mag * (q_fort - 0.60).clip(lower=0.0) + 0.5 * stab_63d.clip(lower=0.0)
    )
    q_knife = (
        -dd_mag * (1.0 - q_fort) * (0.50 + ulcer_pct)
        + 2.0 * cash_burn.clip(upper=0.0)
        + 0.5 * op_chg.clip(upper=0.0)
    )

    s1_in_rk = rec_s1.groupby(level="as_of_date").rank(pct=True).fillna(0.50)
    f_in_rk = rec_f_rk.groupby(level="as_of_date").rank(pct=True).fillna(0.50)
    qf_in_rk = (
        ((q_fort + arch_fcf) / 2.0)
        .groupby(level="as_of_date")
        .rank(pct=True)
        .fillna(0.50)
    )
    sp_in_rk = q_spring.groupby(level="as_of_date").rank(pct=True).fillna(0.50)
    kn_in_rk = q_knife.groupby(level="as_of_date").rank(pct=True).fillna(0.50)

    disq = (kn_in_rk <= 0.20) | ((dd_raw <= -0.35) & (q_fort <= 0.35))
    s3_score = (
        0.45 * s1_in_rk
        + 0.15 * f_in_rk
        + 0.25 * qf_in_rk
        + 0.15 * sp_in_rk
        + np.where(disq, -0.50, 0.0)
    )

    s3_df = pd.DataFrame({"ex": rec_y, "s3": s3_score}, index=rec_X.index).reset_index()
    top15_s3 = (
        s3_df.sort_values(["as_of_date", "s3"], ascending=[True, False])
        .groupby("as_of_date")
        .head(15)
    )

    return {
        "N_eff_Ratio": round(neff_ratio, 3),
        "ROC_AUC": round(ev["ROC_AUC"], 4),
        "PR_AUC": round(ev["PR_AUC"], 4),
        "Recall_>=10%@M": round(ev[f"Recall@Top{int(top_fraction*100)}%"], 4),
        "Tail_Recall_>=25%@M": round(w25_cap / max(w25_tot, 1), 4),
        "Tail_Recall_>=50%@M": round(w50_cap / max(w50_tot, 1), 4),
        "Outliers_>=25%_Cnt": f"{w25_cap}/{w25_tot}",
        "Mega_>=50%_Cnt": f"{w50_cap}/{w50_tot}",
        "Sieve_Spread": round(ev["Sieve_Spread_vs_Pruned"], 4),
        "Sieve_Spread_HAC_SE_11L": round(spread_se, 4),
        "Delta_Sieve_Spread_vs_P0": round(delta_spread, 4),
        "Delta_Sieve_Spread_HAC_SE_11L": round(paired_delta_hac_se, 4),
        "Sieve_Relative_Blowup_Rate": round(ev["Recalled_Tail_Blowup_Rate(<-40%)"], 4),
        "Sieve_Relative_Blowup_Cnt": recalled_blowup_cnt,
        "Mean_Fwd_12M_Excess_Top20_Diag": round(mean_fwd_12m_top20_diag, 4),
        "Exploratory_S3_Proxy_Top15_Fwd12M_Excess": round(float(top15_s3["ex"].mean()), 4),
        "Exploratory_S3_Proxy_Top15_Relative_Blowup_Rate": round(float((top15_s3["ex"] <= -0.40).mean()), 4),
    }


# ------------------------------------------------------------------------------
# 2. FORWARD-ONLY, LABEL-MATURED 4-EXPERT ENSEMBLE OOF GENERATOR
# ------------------------------------------------------------------------------
def compute_forward_only_purged_oof_4expert(
    X_tr: pd.DataFrame,
    y_tr_cont: pd.Series,
    y_tr_bin: pd.Series,
    meta_tr: pd.DataFrame,
    raw_base_weights: np.ndarray = None,
    top_fraction: float = STAGE1_TOP_FRACTION,
) -> pd.DataFrame:
    """Generates strictly forward-only, label-matured OOF predictions for the FULL

    4-Expert Baseline Ensemble (`0.45 Bagged + 0.30 Market + 0.15 Fund + 0.10
    Linear`), ensuring OOF blind spots reflect the actual deployed 4-expert
    ensemble.
    """
    dates = X_tr.index.get_level_values("as_of_date")
    target_end = pd.to_datetime(meta_tr["target_end_date_12m"])
    sup_avail = pd.to_datetime(meta_tr[AVAILABLE])

    block_starts = [
        pd.Timestamp("2018-01-01"),
        pd.Timestamp("2018-07-01"),
        pd.Timestamp("2019-01-01"),
        pd.Timestamp("2019-07-01"),
        pd.Timestamp("2020-01-01"),
        pd.Timestamp("2020-07-01"),
        pd.Timestamp("2021-01-01"),
        pd.Timestamp("2021-07-01"),
        pd.Timestamp("2022-01-01"),
    ]

    oof_ens_score = pd.Series(np.nan, index=X_tr.index, dtype="float64")
    X_ens = X_tr.loc[:, ENSEMBLE_ALL_COLS]
    mono_all = build_monotone_vector(ENSEMBLE_ALL_COLS)
    mono_mkt = build_monotone_vector(ENSEMBLE_MKT_COLS)
    mono_fund = build_monotone_vector(ENSEMBLE_FUND_COLS)

    for b_idx in range(len(block_starts) - 1):
        b_start, b_end = block_starts[b_idx], block_starts[b_idx + 1]
        val_mask = (dates >= b_start) & (dates < b_end)
        if not val_mask.any():
            continue

        # Strict Label-Maturity Purge: every training label must have completed strictly prior to `b_start`
        tr_mask = (
            (dates < b_start) & (target_end < b_start) & (sup_avail < b_start)
        )
        if int(tr_mask.sum()) < 1500:
            continue
        assert (target_end.loc[tr_mask] < b_start).all() and (
            sup_avail.loc[tr_mask] < b_start
        ).all()

        w_sub = None
        if raw_base_weights is not None:
            w_sub = raw_base_weights[tr_mask] / np.mean(raw_base_weights[tr_mask])

        X_tr_b, y_tr_b = X_ens.loc[tr_mask], y_tr_bin.loc[tr_mask]
        X_va_b = X_ens.loc[val_mask]

        cb = LGBMClassifier(**BAGGED_CLF_PARAMS, monotone_constraints=mono_all)
        cb.fit(X_tr_b, y_tr_b, sample_weight=w_sub)
        pb = pd.Series(cb.predict_proba(X_va_b)[:, 1], index=X_va_b.index)

        cm = LGBMClassifier(
            **dict(
                BAGGED_CLF_PARAMS,
                colsample_bytree=0.60,
                monotone_constraints=mono_mkt,
            )
        )
        cm.fit(
            X_tr_b.loc[:, ENSEMBLE_MKT_COLS], y_tr_b, sample_weight=w_sub
        )
        pm = pd.Series(
            cm.predict_proba(X_va_b.loc[:, ENSEMBLE_MKT_COLS])[:, 1],
            index=X_va_b.index,
        )

        cf = LGBMClassifier(
            **dict(
                BAGGED_CLF_PARAMS,
                colsample_bytree=0.50,
                monotone_constraints=mono_fund,
            )
        )
        cf.fit(
            X_tr_b.loc[:, ENSEMBLE_FUND_COLS], y_tr_b, sample_weight=w_sub
        )
        pf = pd.Series(
            cf.predict_proba(X_va_b.loc[:, ENSEMBLE_FUND_COLS])[:, 1],
            index=X_va_b.index,
        )

        _, pl, _ = fit_predict_linear_rank_expert_stable(X_tr_b, y_tr_b, X_va_b)

        ens_b = (
            0.45 * to_cs_rank(pb)
            + 0.30 * to_cs_rank(pm)
            + 0.15 * to_cs_rank(pf)
            + 0.10 * to_cs_rank(pl)
        )
        oof_ens_score.loc[val_mask] = ens_b

    oof_df = pd.DataFrame(
        {"actual_excess": y_tr_cont, "y_bin": y_tr_bin, "oof_score": oof_ens_score},
        index=X_tr.index,
    )
    valid_oof = oof_df["oof_score"].notna()
    oof_df["oof_cs_rank"] = np.nan
    oof_df.loc[valid_oof, "oof_cs_rank"] = (
        oof_df.loc[valid_oof]
        .groupby(level="as_of_date")["oof_score"]
        .rank(pct=True, ascending=True)
    )

    oof_df["is_oof_missed_winner"] = (
        valid_oof
        & (oof_df["actual_excess"] >= 0.20)
        & (oof_df["oof_cs_rank"] < (1.0 - top_fraction))
    )
    oof_df["is_oof_false_pos_blowup"] = (
        valid_oof
        & (oof_df["actual_excess"] <= -0.30)
        & (oof_df["oof_cs_rank"] >= (1.0 - top_fraction))
    )
    oof_df["raw_hard_mult"] = np.where(
        oof_df["is_oof_missed_winner"] | oof_df["is_oof_false_pos_blowup"],
        1.45,
        1.00,
    )
    return oof_df


# ------------------------------------------------------------------------------
# 3. EXECUTE PHASES 0 THROUGH 5 SEQUENTIALLY ACROSS BOTH DEVELOPMENT FOLDS
# ------------------------------------------------------------------------------
COMPOUNDER_5TH_MONO_MAP = {
    k: v
    for k, v in STRICT_MONOTONE_MAP.items()
    if not k.startswith("price_drawdown")
}
COMPOUNDER_5TH_MONO_MAP.update({
    "fund_fortress_quality_cs_score": 1,
    "fund_peer_archetype_fcf_rank": 1,
    "fund_peer_archetype_rev_share_gain": 1,
    "fund_productive_capex_compounder": 1,
    "price_momentum_12m_ex_1m": 1,
    "price_momentum_12m_ex_1m_cs_pct": 1,
    "price_martin_ratio_252d": 1,
    "price_martin_ratio_252d_cs_pct": 1,
})
# Prevent duplicate LightGBM feature names if fortress quality is already present.
FIFTH_EXPERT_COLS = list(dict.fromkeys(ENSEMBLE_ALL_COLS + ["fund_fortress_quality_cs_score"]))
assert len(FIFTH_EXPERT_COLS) == len(set(FIFTH_EXPERT_COLS))

phase0_audit_rows = []
phased_results_rows = []
oot_learnability_rows = []

for fold_id in FOLD_IDS:
    fdata = get_fold(fold_id)
    X_tr = fdata["train"]["X"]
    y_tr_cont = fdata["train"]["y"]
    y_tr_bin = fdata["train"]["y_bin"]
    meta_tr = fdata["train"]["metadata"]
    X_va = fdata["validation"]["X"]
    y_va_cont = fdata["validation"]["y"]
    y_va_bin = fdata["validation"]["y_bin"]
    meta_va = fdata["validation"]["metadata"]
    assert X_tr.index.equals(y_tr_cont.index) and X_tr.index.equals(y_tr_bin.index)
    assert X_tr.index.equals(meta_tr.index) and X_va.index.equals(meta_va.index)

    # ==========================================================================
    # PHASE 0: OUTER-FOLD LABEL-MATURITY PROOF & FROZEN v26 BASELINE PARITY
    # ==========================================================================
    tr_dates = X_tr.index.get_level_values("as_of_date")
    va_dates = X_va.index.get_level_values("as_of_date")
    val_start = va_dates.min()
    tr_target_end = pd.to_datetime(meta_tr["target_end_date_12m"])
    tr_sup_avail = pd.to_datetime(meta_tr[AVAILABLE])

    assert (tr_dates < val_start).all(), f"Fold {fold_id}: training as_of_date >= val_start!"
    assert (tr_target_end < val_start).all(), f"Fold {fold_id}: training target_end_date_12m >= val_start!"
    assert (tr_sup_avail < val_start).all(), f"Fold {fold_id}: training supervision_available_date >= val_start!"

    X_tr_ens = X_tr.loc[:, ENSEMBLE_ALL_COLS]
    X_va_ens = X_va.loc[:, ENSEMBLE_ALL_COLS]
    mono_all = build_monotone_vector(ENSEMBLE_ALL_COLS)
    mono_mkt = build_monotone_vector(ENSEMBLE_MKT_COLS)
    mono_fund = build_monotone_vector(ENSEMBLE_FUND_COLS)
    mono_5th = build_monotone_vector(FIFTH_EXPERT_COLS, COMPOUNDER_5TH_MONO_MAP)

    _, p_lin, _ = fit_predict_linear_rank_expert_stable(X_tr_ens, y_tr_bin, X_va_ens)
    r_lin = to_cs_rank(p_lin)

    def fit_4expert_ensemble(w_norm_vec: np.ndarray = None):
        cb = LGBMClassifier(**BAGGED_CLF_PARAMS, monotone_constraints=mono_all)
        cb.fit(X_tr_ens, y_tr_bin, sample_weight=w_norm_vec)
        rb = to_cs_rank(pd.Series(cb.predict_proba(X_va_ens)[:, 1], index=X_va.index))

        cm = LGBMClassifier(
            **dict(BAGGED_CLF_PARAMS, colsample_bytree=0.60, monotone_constraints=mono_mkt)
        )
        cm.fit(X_tr_ens.loc[:, ENSEMBLE_MKT_COLS], y_tr_bin, sample_weight=w_norm_vec)
        rm = to_cs_rank(
            pd.Series(cm.predict_proba(X_va_ens.loc[:, ENSEMBLE_MKT_COLS])[:, 1], index=X_va.index)
        )

        cf = LGBMClassifier(
            **dict(BAGGED_CLF_PARAMS, colsample_bytree=0.50, monotone_constraints=mono_fund)
        )
        cf.fit(X_tr_ens.loc[:, ENSEMBLE_FUND_COLS], y_tr_bin, sample_weight=w_norm_vec)
        rf = to_cs_rank(
            pd.Series(cf.predict_proba(X_va_ens.loc[:, ENSEMBLE_FUND_COLS])[:, 1], index=X_va.index)
        )

        ens = 0.45 * rb + 0.30 * rm + 0.15 * rf + 0.10 * r_lin
        return ens, rb, rm, rf

    p_P0_control, rb_flat, rm_flat, rf_flat = fit_4expert_ensemble(None)
    frozen_v26_prob = stage1_probabilities[(fold_id, CANONICAL_STAGE1_MODEL)]
    assert p_P0_control.index.equals(frozen_v26_prob.index), "Frozen baseline index/order mismatch"
    parity_diff = float(np.max(np.abs(p_P0_control.to_numpy() - frozen_v26_prob.to_numpy())))
    assert parity_diff < 1e-12, f"Phase 0 Baseline Parity Failed on Fold {fold_id}: {parity_diff}"

    phase0_audit_rows.append({
        "fold": fold_id,
        "train_rows": len(X_tr),
        "last_train_as_of_date": tr_dates.max().strftime("%Y-%m-%d"),
        "last_train_label_maturity_date": tr_target_end.max().strftime("%Y-%m-%d"),
        "first_validation_as_of_date": val_start.strftime("%Y-%m-%d"),
        "outer_purge_gap_days": int((val_start - tr_target_end.max()).days),
        "frozen_v26_max_abs_diff": f"{parity_diff:.2e}",
        "outer_purge_and_parity_verified": True,
    })

    phased_results_rows.append({
        "fold": fold_id,
        "phase_experiment": "Phase 0: Certified v26 Baseline Control (4-Expert, Flat Weights)",
        **evaluate_stage1_and_downstream_stage3(
            X_va, y_va_cont, y_va_bin, p_P0_control, rf_flat, neff_ratio=1.000
        ),
    })

    # ==========================================================================
    # PHASE 1: COST-SENSITIVE PAYOFF WEIGHTING ONLY (NO OOF MINING, NO 5TH EXPERT)
    # ==========================================================================
    raw_w_cost = compute_raw_cost_weights(y_tr_cont, w_cap=W_MAX_CAP)
    w_P1_norm, neff_P1 = combine_and_normalize_weights_once(raw_w_cost, None, w_cap=W_MAX_CAP)
    p_P1_cost, rb_cost, rm_cost, rf_cost = fit_4expert_ensemble(w_P1_norm)

    phased_results_rows.append({
        "fold": fold_id,
        "phase_experiment": "Phase 1: Cost-Sensitive Payoff Weighting Only (4-Expert)",
        **evaluate_stage1_and_downstream_stage3(
            X_va, y_va_cont, y_va_bin, p_P1_cost, rf_flat, neff_ratio=neff_P1,
            baseline_stage1_score=p_P0_control
        ),
    })

    # ==========================================================================
    # PHASE 2: DIRECT CONCAVE UTILITY REGRESSION U(r) — L2 (E[U|X]) VS. HUBER
    # ==========================================================================
    y_tr_util = compute_direct_economic_utility(y_tr_cont)

    # P2a: L2 Squared-Error Regression on U(r) (estimates true conditional expectation E[U(R)|X])
    L2_UTIL_PARAMS = dict(
        objective="regression",
        n_estimators=250,
        learning_rate=0.025,
        num_leaves=15,
        min_child_samples=120,
        colsample_bytree=0.45,
        subsample=0.75,
        subsample_freq=1,
        reg_lambda=5.0,
        random_state=SEED,
        n_jobs=4,
        verbose=-1,
    )
    reg_util_l2 = LGBMRegressor(**L2_UTIL_PARAMS, monotone_constraints=mono_all)
    reg_util_l2.fit(X_tr_ens, y_tr_util)
    r_util_l2 = to_cs_rank(pd.Series(reg_util_l2.predict(X_va_ens), index=X_va.index))

    # P2b: Huber Regression on U(r) (robust conditional location)
    reg_util_huber = LGBMRegressor(**dict(HUBER_REG_PARAMS, monotone_constraints=mono_all))
    reg_util_huber.fit(X_tr_ens, y_tr_util)
    r_util_huber = to_cs_rank(pd.Series(reg_util_huber.predict(X_va_ens), index=X_va.index))

    for p2_name, p2_score in [
        ("Phase 2a: Direct Utility U(r) Standalone — L2 Squared-Error (E[U|X])", r_util_l2),
        ("Phase 2b: Direct Utility U(r) Standalone — Huber Loss", r_util_huber),
        ("Phase 2c: 4-Expert + L2 Utility U(r) Blend (85/15)", 0.85 * p_P0_control + 0.15 * r_util_l2),
        ("Phase 2d: 4-Expert + Huber Utility U(r) Blend (85/15)", 0.85 * p_P0_control + 0.15 * r_util_huber),
    ]:
        phased_results_rows.append({
            "fold": fold_id,
            "phase_experiment": p2_name,
            **evaluate_stage1_and_downstream_stage3(
                X_va, y_va_cont, y_va_bin, p2_score, rf_flat, neff_ratio=1.000,
                baseline_stage1_score=p_P0_control
            ),
        })

    # ==========================================================================
    # PHASE 3: FORWARD-ONLY 4-EXPERT PURGED OOF MINING & OUT-OF-TIME LEARNABILITY TEST
    # ==========================================================================
    oof_4exp_df = compute_forward_only_purged_oof_4expert(
        X_tr, y_tr_cont, y_tr_bin, meta_tr, raw_base_weights=None
    )

    # Matched OOT test: identical specialist, same data/cutoff, flat vs. OOF-hard weights.
    # Early OOF error labels must mature before 2020-07-01; test later rows only.
    early_oof_mask = (
        oof_4exp_df["oof_cs_rank"].notna()
        & (tr_dates >= pd.Timestamp("2018-01-01"))
        & (tr_dates <= pd.Timestamp("2019-06-30"))
        & (tr_target_end < pd.Timestamp("2020-07-01"))
        & (tr_sup_avail < pd.Timestamp("2020-07-01"))
    )
    late_oof_mask = (
        oof_4exp_df["oof_cs_rank"].notna()
        & (tr_dates >= pd.Timestamp("2020-07-01"))
    )
    if early_oof_mask.any() and late_oof_mask.any():
        # Fixed historical cutoff. Change only the specialist's sample weights.
        pre_2020_mask = (
            (tr_dates < pd.Timestamp("2020-07-01"))
            & (tr_target_end < pd.Timestamp("2020-07-01"))
            & (tr_sup_avail < pd.Timestamp("2020-07-01"))
        )
        w_oot_raw = oof_4exp_df.loc[pre_2020_mask, "raw_hard_mult"].to_numpy(dtype=np.float64)
        w_oot_norm = w_oot_raw / np.mean(w_oot_raw)

        X_early = X_tr_ens.loc[pre_2020_mask]
        y_early = y_tr_bin.loc[pre_2020_mask]
        X_late = X_tr_ens.loc[late_oof_mask]
        c_oot_flat = LGBMClassifier(**BAGGED_CLF_PARAMS, monotone_constraints=mono_all)
        c_oot_hard = LGBMClassifier(**BAGGED_CLF_PARAMS, monotone_constraints=mono_all)
        c_oot_flat.fit(X_early, y_early)
        c_oot_hard.fit(X_early, y_early, sample_weight=w_oot_norm)
        p_flat_late = to_cs_rank(pd.Series(c_oot_flat.predict_proba(X_late)[:, 1], index=X_late.index))
        p_hard_late = to_cs_rank(pd.Series(c_oot_hard.predict_proba(X_late)[:, 1], index=X_late.index))

        # Common baseline and blend coefficient for both treatments.
        base_late_rk = oof_4exp_df.loc[late_oof_mask, "oof_cs_rank"]
        flat_blend = (0.70 * base_late_rk + 0.30 * p_flat_late).groupby(level="as_of_date").rank(pct=True)
        hard_blend = (0.70 * base_late_rk + 0.30 * p_hard_late).groupby(level="as_of_date").rank(pct=True)

        y_late_cont = y_tr_cont.loc[late_oof_mask]
        w25_late_tot = int((y_late_cont >= 0.25).sum())
        base_w25_rec = float((y_late_cont.loc[base_late_rk >= 0.70] >= 0.25).sum() / max(w25_late_tot, 1))
        flat_w25_rec = float((y_late_cont.loc[flat_blend >= 0.70] >= 0.25).sum() / max(w25_late_tot, 1))
        hard_w25_rec = float((y_late_cont.loc[hard_blend >= 0.70] >= 0.25).sum() / max(w25_late_tot, 1))

        oot_learnability_rows.append({
            "fold": fold_id,
            "early_matured_train_rows": int(pre_2020_mask.sum()),
            "later_oot_test_rows": int(late_oof_mask.sum()),
            "late_oof_4expert_Recall_>=25%": round(base_w25_rec, 4),
            "late_flat_specialist_blend_Recall_>=25%": round(flat_w25_rec, 4),
            "late_hard_specialist_blend_Recall_>=25%": round(hard_w25_rec, 4),
            "delta_OOT_hard_minus_flat_Recall_>=25%": round(hard_w25_rec - flat_w25_rec, 4),
        })

    w_P3_norm, neff_P3 = combine_and_normalize_weights_once(
        np.ones(len(X_tr), dtype=np.float64),
        oof_4exp_df["raw_hard_mult"].to_numpy(dtype=np.float64),
        w_cap=W_MAX_CAP,
    )
    p_P3_oof, rb_oof, rm_oof, rf_oof = fit_4expert_ensemble(w_P3_norm)

    phased_results_rows.append({
        "fold": fold_id,
        "phase_experiment": "Phase 3: Forward-Only 4-Expert Purged OOF Mining Only",
        **evaluate_stage1_and_downstream_stage3(
            X_va, y_va_cont, y_va_bin, p_P3_oof, rf_flat, neff_ratio=neff_P3,
            baseline_stage1_score=p_P0_control
        ),
    })

    # ==========================================================================
    # PHASE 4: 5TH COMPOUNDER SPECIALIST (UNCONSTRAINED DRAWDOWN) — STANDALONE & MARGINAL
    # ==========================================================================
    c5_flat = LGBMClassifier(
        **dict(BAGGED_CLF_PARAMS, colsample_bytree=0.50, monotone_constraints=mono_5th)
    )
    c5_flat.fit(X_tr.loc[:, FIFTH_EXPERT_COLS], y_tr_bin)
    r_c5_flat = to_cs_rank(
        pd.Series(c5_flat.predict_proba(X_va.loc[:, FIFTH_EXPERT_COLS])[:, 1], index=X_va.index)
    )
    p_P4_ens = 0.38 * rb_flat + 0.25 * rm_flat + 0.12 * rf_flat + 0.15 * r_c5_flat + 0.10 * r_lin

    for p4_name, p4_score in [
        ("Phase 4a: 5th Compounder Specialist Standalone (Flat Weights)", r_c5_flat),
        ("Phase 4b: 4-Expert + 5th Compounder Specialist (85/15 Blend, Flat Weights)", p_P4_ens),
    ]:
        phased_results_rows.append({
            "fold": fold_id,
            "phase_experiment": p4_name,
            **evaluate_stage1_and_downstream_stage3(
                X_va, y_va_cont, y_va_bin, p4_score, rf_flat, neff_ratio=1.000,
                baseline_stage1_score=p_P0_control
            ),
        })

    # ==========================================================================
    # PHASE 5: CURATED TWO-WAY INTERACTIONS (CORRECTED SINGLE-CLIP NORMALIZATION)
    # ==========================================================================
    # Interaction 5a: Cost-Sensitive Weighting + 4-Expert Purged OOF Mining (Unnormalized multiply -> single clip -> single norm)
    oof_4exp_cost_df = compute_forward_only_purged_oof_4expert(
        X_tr, y_tr_cont, y_tr_bin, meta_tr, raw_base_weights=raw_w_cost
    )
    w_P5a_norm, neff_P5a = combine_and_normalize_weights_once(
        raw_w_cost,
        oof_4exp_cost_df["raw_hard_mult"].to_numpy(dtype=np.float64),
        w_cap=W_MAX_CAP,
    )
    p_P5a_comb, rb_c_oof, rm_c_oof, rf_c_oof = fit_4expert_ensemble(w_P5a_norm)

    # Interaction 5b: Cost-Sensitive Weighting + L2 Direct Utility U(r) Blend
    p_P5b_cost_util = 0.85 * p_P1_cost + 0.15 * r_util_l2

    # Interaction 5c: Cost-Sensitive Weighting + 5th Compounder Specialist
    c5_cost = LGBMClassifier(
        **dict(BAGGED_CLF_PARAMS, colsample_bytree=0.50, monotone_constraints=mono_5th)
    )
    c5_cost.fit(X_tr.loc[:, FIFTH_EXPERT_COLS], y_tr_bin, sample_weight=w_P1_norm)
    r_c5_cost = to_cs_rank(
        pd.Series(c5_cost.predict_proba(X_va.loc[:, FIFTH_EXPERT_COLS])[:, 1], index=X_va.index)
    )
    p_P5c_cost_5th = 0.38 * rb_cost + 0.25 * rm_cost + 0.12 * rf_cost + 0.15 * r_c5_cost + 0.10 * r_lin

    for p5_name, p5_score, p5_neff in [
        ("Phase 5a: Cost-Sensitive Weighting + 4-Expert Purged OOF", p_P5a_comb, neff_P5a),
        ("Phase 5b: Cost-Sensitive Weighting + L2 Utility U(r) Blend", p_P5b_cost_util, neff_P1),
        ("Phase 5c: Cost-Sensitive Weighting + 5th Compounder Specialist", p_P5c_cost_5th, neff_P1),
    ]:
        phased_results_rows.append({
            "fold": fold_id,
            "phase_experiment": p5_name,
            **evaluate_stage1_and_downstream_stage3(
                X_va, y_va_cont, y_va_bin, p5_score, rf_flat, neff_ratio=p5_neff,
                baseline_stage1_score=p_P0_control
            ),
        })

print("=== PHASE 0: OUTER-FOLD LABEL-MATURITY PURGE & FROZEN v26 BASELINE PARITY AUDIT ===")
display(pd.DataFrame(phase0_audit_rows).set_index("fold"))

print("\n=== PHASE 3 DIAGNOSTIC: MATCHED OUT-OF-TIME HARD-MINING TEST (FLAT VS HARD SPECIALIST) ===")
if oot_learnability_rows:
    display(pd.DataFrame(oot_learnability_rows).set_index("fold"))
else:
    print("No mature/OOF-scored early-late split in these development folds; diagnostic skipped.")

phased_df = pd.DataFrame(phased_results_rows).set_index(["fold", "phase_experiment"])
print("\n=== SEQUENTIAL PHASED ABLATION SCORECARD (PHASES 0–5: SIEVE + EXPLORATORY STAGE 3 PROXY) ===")
display(phased_df)

=== PHASE 0: OUTER-FOLD LABEL-MATURITY PURGE & FROZEN v26 BASELINE PARITY AUDIT ===


,train_rows,last_train_as_of_date,last_train_label_maturity_date,first_validation_as_of_date,outer_purge_gap_days,frozen_v26_max_abs_diff,outer_purge_and_parity_verified
fold,,,,,,,
0,28968,2021-06-30,2022-06-30,2022-07-29,29,0.00e+00,True
1,31940,2021-12-31,2022-12-30,2023-01-31,32,0.00e+00,True



=== PHASE 3 DIAGNOSTIC: MATCHED OUT-OF-TIME HARD-MINING TEST (FLAT VS HARD SPECIALIST) ===


,early_matured_train_rows,later_oot_test_rows,late_oof_4expert_Recall_>=25%,late_flat_specialist_blend_Recall_>=25%,late_hard_specialist_blend_Recall_>=25%,delta_OOT_hard_minus_flat_Recall_>=25%
fold,,,,,,
0,17461,5928,0.2882,0.2882,0.2773,-0.0108
1,17461,8900,0.2631,0.2649,0.2584,-0.0065



=== SEQUENTIAL PHASED ABLATION SCORECARD (PHASES 0–5: SIEVE + EXPLORATORY STAGE 3 PROXY) ===


N_eff_Ratio  ROC_AUC  \
fold phase_experiment                                                           
0    Phase 0: Certified v26 Baseline Control (4-Expe...        1.000   0.6618   
     Phase 1: Cost-Sensitive Payoff Weighting Only (...        0.903   0.6610   
     Phase 2a: Direct Utility U(r) Standalone — L2 S...        1.000   0.6101   
     Phase 2b: Direct Utility U(r) Standalone — Hube...        1.000   0.6092   
     Phase 2c: 4-Expert + L2 Utility U(r) Blend (85/15)        1.000   0.6573   
     Phase 2d: 4-Expert + Huber Utility U(r) Blend (...        1.000   0.6569   
     Phase 3: Forward-Only 4-Expert Purged OOF Minin...        0.982   0.6584   
     Phase 4a: 5th Compounder Specialist Standalone ...        1.000   0.6594   
     Phase 4b: 4-Expert + 5th Compounder Specialist ...        1.000   0.6627   
     Phase 5a: Cost-Sensitive Weighting + 4-Expert P...        0.855   0.6572   
     Phase 5b: Cost-Sensitive Weighting + L2 Utility...        0.903   0.6558   
     Phase 5c: Cost-Sensitive Weighting + 5th Compou...        0.903   0.6617   
1    Phase 0: Certified v26 Baseline Control (4-Expe...        1.000   0.5848   
     Phase 1: Cost-Sensitive Payoff Weighting Only (...        0.902   0.6010   
     Phase 2a: Direct Utility U(r) Standalone — L2 S...        1.000   0.5941   
     Phase 2b: Direct Utility U(r) Standalone — Hube...        1.000   0.5781   
     Phase 2c: 4-Expert + L2 Utility U(r) Blend (85/15)        1.000   0.5892   
     Phase 2d: 4-Expert + Huber Utility U(r) Blend (...        1.000   0.5866   
     Phase 3: Forward-Only 4-Expert Purged OOF Minin...        0.981   0.5856   
     Phase 4a: 5th Compounder Specialist Standalone ...        1.000   0.5666   
     Phase 4b: 4-Expert + 5th Compounder Specialist ...        1.000   0.5828   
     Phase 5a: Cost-Sensitive Weighting + 4-Expert P...        0.851   0.5941   
     Phase 5b: Cost-Sensitive Weighting + L2 Utility...        0.902   0.6025   
     Phase 5c: Cost-Sensitive Weighting + 5th Compou...        0.902   0.5976   

                                                         PR_AUC  \
fold phase_experiment                                             
0    Phase 0: Certified v26 Baseline Control (4-Expe...  0.3270   
     Phase 1: Cost-Sensitive Payoff Weighting Only (...  0.3234   
     Phase 2a: Direct Utility U(r) Standalone — L2 S...  0.2823   
     Phase 2b: Direct Utility U(r) Standalone — Hube...  0.2867   
     Phase 2c: 4-Expert + L2 Utility U(r) Blend (85/15)  0.3237   
     Phase 2d: 4-Expert + Huber Utility U(r) Blend (...  0.3237   
     Phase 3: Forward-Only 4-Expert Purged OOF Minin...  0.3150   
     Phase 4a: 5th Compounder Specialist Standalone ...  0.3228   
     Phase 4b: 4-Expert + 5th Compounder Specialist ...  0.3293   
     Phase 5a: Cost-Sensitive Weighting + 4-Expert P...  0.3178   
     Phase 5b: Cost-Sensitive Weighting + L2 Utility...  0.3188   
     Phase 5c: Cost-Sensitive Weighting + 5th Compou...  0.3247   
1    Phase 0: Certified v26 Baseline Control (4-Expe...  0.2727   
     Phase 1: Cost-Sensitive Payoff Weighting Only (...  0.2845   
     Phase 2a: Direct Utility U(r) Standalone — L2 S...  0.2771   
     Phase 2b: Direct Utility U(r) Standalone — Hube...  0.2634   
     Phase 2c: 4-Expert + L2 Utility U(r) Blend (85/15)  0.2756   
     Phase 2d: 4-Expert + Huber Utility U(r) Blend (...  0.2726   
     Phase 3: Forward-Only 4-Expert Purged OOF Minin...  0.2717   
     Phase 4a: 5th Compounder Specialist Standalone ...  0.2612   
     Phase 4b: 4-Expert + 5th Compounder Specialist ...  0.2726   
     Phase 5a: Cost-Sensitive Weighting + 4-Expert P...  0.2798   
     Phase 5b: Cost-Sensitive Weighting + L2 Utility...  0.2846   
     Phase 5c: Cost-Sensitive Weighting + 5th Compou...  0.2831   

                                                         Recall_>=10%@M  \
fold phase_experiment                                                     
0    Phase 0: Certified v26 Baseline Control (4-Expe...  

In [19]:
# === PERSONA NOTEBOOK 04: APPEND-ONLY, AUDITED V30 EXPLANATION HANDOFF ===
# Paste into ONE new cell after the last completed Experiment H4 cell.
# Does NOT change model selection, H4 signals, or validation cohorts.
# Replays the exact H1 LightGBM LambdaRank recipe from this notebook and
# refuses to export SHAP unless its within-month ranks match frozen H4.
# Core contributions below are exact WEIGHTED MODEL-FAMILY RANKS, not SHAP.
# Core feature-level SHAP still requires fitted model/preprocessor recovery.

from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from lightgbm import LGBMRanker

assert all(name in globals() for name in (
    "FOLD_IDS", "get_fold", "ENSEMBLE_ALL_COLS", "h_predictions",
    "h4_handoff", "g_validation_predictions", "four_family_val_predictions",
    "g_base_components", "g_rank", "h_cs_rank", "h_relevance",
    "H_RANKER_TREES", "H_GAINS", "SEED", "AVAILABLE", "H4_OUTDIR",
)), "Run Notebook 04 through H4 in this kernel before exporting."

EXPORT_ROOT_V30 = Path(H4_OUTDIR) / "explainability_v30_handoff"
PUBLIC_V30 = EXPORT_ROOT_V30 / "application_safe"
RESEARCH_V30 = EXPORT_ROOT_V30 / "RESEARCH_ONLY_DO_NOT_SERVE"
PUBLIC_V30.mkdir(parents=True, exist_ok=True)
RESEARCH_V30.mkdir(parents=True, exist_ok=True)

CORE_WEIGHTS_V30 = {"lgbm": 0.25, "rf": 0.25, "lin": 0.25,
                    "mlp": 0.125, "identity_bce_2": 0.125}
assert abs(sum(CORE_WEIGHTS_V30.values()) - 1.0) < 1e-12


def _v30_replay_ranker(X_tr, y_tr):
    """Match h_fit_lambdarank training exactly; refit, don't reuse labels as features."""
    assert X_tr.index.equals(y_tr.index)
    dates = pd.DatetimeIndex(X_tr.index.get_level_values("as_of_date"))
    order = np.argsort(dates.values, kind="stable")
    X_fit = X_tr.iloc[order].replace([np.inf, -np.inf], np.nan)
    y_fit = h_relevance(y_tr.iloc[order])
    dates_sorted = dates[order]
    groups = dates_sorted.value_counts(sort=False).sort_index().to_numpy(dtype=np.int32)
    assert groups.sum() == len(X_fit)
    if np.unique(y_fit).size < 2:
        raise ValueError("No relevance variation in ranker training labels")
    ranker = LGBMRanker(
        objective="lambdarank", metric="ndcg", n_estimators=H_RANKER_TREES,
        learning_rate=0.035, num_leaves=15, max_depth=5,
        min_child_samples=70, colsample_bytree=0.80,
        subsample=0.85, subsample_freq=1, reg_lambda=4.0,
        lambdarank_truncation_level=8,
        label_gain=H_GAINS.astype(int).tolist(),
        random_state=int(SEED), n_jobs=-1, verbosity=-1,
    )
    ranker.fit(X_fit, y_fit, group=groups)
    return ranker


def _v30_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


manifest_v30 = {
    "schema": "persona_nb04_v30_explainability_export_1",
    "purpose": "Historical research attribution; no forward validation labels exported",
    "core_component_weights": CORE_WEIGHTS_V30,
    "core_component_semantics": "weighted_monthly_percentile_ranks; NOT SHAP",
    "lambda_shap_semantics": "LightGBM TreeSHAP contribution to RAW LambdaRank score, NOT percentage return or percentile rank",
    "full_core_shap_available": False,
    "folds": {},
    "file_sha256": {},
}

for fold in FOLD_IDS:
    fd = get_fold(fold)
    X_tr = fd["train"]["X"].loc[:, ENSEMBLE_ALL_COLS]
    X_va = fd["validation"]["X"].loc[:, ENSEMBLE_ALL_COLS]
    y_tr = fd["train"]["y"]
    meta = fd["train"]["metadata"]
    frozen = h4_handoff[fold]

    assert X_tr.index.equals(y_tr.index)
    assert X_tr.index.equals(meta.index)
    assert X_va.index.equals(frozen.index), "H4/export validation index mismatch"
    assert X_va.index.is_unique and X_tr.index.is_unique
    assert list(X_tr.columns) == list(X_va.columns)
    assert "core20_member" in frozen and "convex_top3" in frozen
    assert int(frozen["core20_member"].sum()) == 120, "Expected 6 months x 20 candidates"
    assert int(frozen["convex_top3"].sum()) == 18, "Expected 6 months x 3 candidates"
    first_val_date = pd.Timestamp(X_va.index.get_level_values("as_of_date").min())
    assert (pd.to_datetime(meta["target_end_date_12m"]) < first_val_date).all()
    assert (pd.to_datetime(meta[AVAILABLE]) < first_val_date).all()

    # 1. Recover exact five FAMILY-LEVEL core scoring components from cache.
    comp = g_base_components(fold, X_va.index)
    comp["identity_bce_2"] = g_rank(
        g_validation_predictions[fold]["fresh"]["identity_bce_2"]
    )
    core_unranked = sum(CORE_WEIGHTS_V30[name] * comp[name]
                        for name in CORE_WEIGHTS_V30)
    core_percentile = h_cs_rank(core_unranked)
    core_max_error = float(np.max(np.abs(
        core_percentile.to_numpy() - frozen["core"].to_numpy()
    )))
    assert core_max_error < 1e-12, (
        f"Fold {fold}: CORE parity failure {core_max_error:.3g}. No export."
    )

    # 2. Replay the frozen H1 model. Verify full-universe rank parity first.
    ranker = _v30_replay_ranker(X_tr, y_tr)
    X_va_safe = X_va.replace([np.inf, -np.inf], np.nan)
    ranker_raw = pd.Series(ranker.predict(X_va_safe),
                           index=X_va.index, name="lambda_raw_score")
    assert np.isfinite(ranker_raw.to_numpy()).all()
    ranker_pct = h_cs_rank(ranker_raw)
    lambda_max_error = float(np.max(np.abs(
        ranker_pct.to_numpy() - frozen["H1_LambdaRank"].to_numpy()
    )))
    assert lambda_max_error < 1e-12, (
        f"Fold {fold}: LAMBDARANK parity failure {lambda_max_error:.3g}. "
        "This is not the identical frozen ranker; do NOT label its SHAP as faithful."
    )

    # 3. Restrict application export to EXACT H4 core20 issuer-month members.
    select = frozen["core20_member"].astype(bool).to_numpy()
    X20 = X_va_safe.loc[select]
    frame = frozen.loc[select].copy()
    frame["fold"] = int(fold)
    frame["lambda_raw_score"] = ranker_raw.loc[select]
    frame["core_pre_percentile_blend"] = core_unranked.loc[select]
    frame["core_percentile_replayed"] = core_percentile.loc[select]

    for name, weight in CORE_WEIGHTS_V30.items():
        frame[f"core_{name}_month_percentile"] = comp[name].loc[select]
        frame[f"core_{name}_weighted_centered_contribution"] = (
            weight * (comp[name].loc[select] - 0.5)
        )

    # Numeric identity: weighted deviations sum to (pre-rank blend - 0.5).
    centered = sum(frame[f"core_{name}_weighted_centered_contribution"]
                   for name in CORE_WEIGHTS_V30)
    assert np.allclose(centered.to_numpy(),
                       (frame["core_pre_percentile_blend"] - 0.5).to_numpy(),
                       rtol=0, atol=1e-12)

    # 4. GENUINE TreeSHAP contributions for the replayed LambdaRank model.
    # Includes base value as the final column. All contributions are RAW-score units.
    contribution = np.asarray(ranker.booster_.predict(X20, pred_contrib=True))
    expected_shape = (len(X20), len(ENSEMBLE_ALL_COLS) + 1)
    assert contribution.shape == expected_shape, (
        f"Unexpected LightGBM SHAP tensor shape: {contribution.shape}"
    )
    raw20 = ranker_raw.loc[select].to_numpy(dtype=float)
    shap_error = float(np.max(np.abs(contribution.sum(axis=1) - raw20)))
    assert shap_error < 1e-5, (
        f"Fold {fold}: SHAP additivity error {shap_error:.4g}"
    )
    frame["lambda_shap_expected_value"] = contribution[:, -1]

    shap_matrix = pd.DataFrame(contribution[:, :-1], index=X20.index,
                               columns=ENSEMBLE_ALL_COLS)
    pos_reasons = []
    neg_reasons = []
    for row_id in range(len(X20)):
        attributions = contribution[row_id, :-1]
        positive = np.flatnonzero(attributions > 0)
        negative = np.flatnonzero(attributions < 0)
        positive = positive[np.argsort(-attributions[positive], kind="stable")][:3]
        negative = negative[np.argsort(attributions[negative], kind="stable")][:3]
        for label, indexes, target in (
            ("positive", positive, pos_reasons),
            ("negative", negative, neg_reasons),
        ):
            for j, k in enumerate(indexes, 1):
                idx = X20.index[row_id]
                target.append({
                    "fold": int(fold),
                    "as_of_date": str(pd.Timestamp(idx[X20.index.names.index("as_of_date")]).date()),
                    "security_key": str(idx[X20.index.names.index("security_key")]),
                    "direction": label,
                    "importance_rank": j,
                    "feature": str(ENSEMBLE_ALL_COLS[k]),
                    "feature_value": float(X20.iloc[row_id, k]) if pd.notna(X20.iloc[row_id, k]) else np.nan,
                    "lambda_shap_raw_score_units": float(attributions[k]),
                })

    frame = frame.reset_index()
    frame.insert(0, "source_contract", "H4_historical_label_free")

    safe_path = PUBLIC_V30 / f"fold{fold}_core20_scoring_contributions.parquet"
    frame.to_parquet(safe_path, index=False)
    top_path = PUBLIC_V30 / f"fold{fold}_lambda_shap_top_drivers.parquet"
    pd.DataFrame(pos_reasons + neg_reasons).to_parquet(top_path, index=False)
    full_shap_path = RESEARCH_V30 / f"fold{fold}_lambda_shap_full.parquet"
    shap_matrix.to_parquet(full_shap_path)
    ranker_path = RESEARCH_V30 / f"fold{fold}_frozen_parity_lambdarank.txt"
    ranker.booster_.save_model(str(ranker_path))

    # 5. Inputs for later faithful reconstruction of Core tree/linear/neural models.
    # These stay separate from application-safe, label-free exports.
    xtr_path = RESEARCH_V30 / f"fold{fold}_X_train.parquet"
    X_tr.to_parquet(xtr_path)
    ytr_path = RESEARCH_V30 / f"fold{fold}_training_supervision.parquet"
    pd.concat([
        y_tr.rename("train_forward_12m_excess"),
        fd["train"]["y_bin"].rename("train_binary_label"),
        meta[["target_end_date_12m", AVAILABLE]],
    ], axis=1).to_parquet(ytr_path)
    xva_path = RESEARCH_V30 / f"fold{fold}_X_validation_features_only.parquet"
    X_va.to_parquet(xva_path)

    files = (safe_path, top_path, full_shap_path, ranker_path,
             xtr_path, ytr_path, xva_path)
    for p in files:
        manifest_v30["file_sha256"][str(p.relative_to(EXPORT_ROOT_V30))] = _v30_sha(p)
    manifest_v30["folds"][str(fold)] = {
        "first_validation_date": str(first_val_date.date()),
        "core20_rows": int(len(frame)),
        "lambda_shap_max_additivity_error": shap_error,
        "lambda_pct_rank_max_abs_parity_error": lambda_max_error,
        "core_pct_rank_max_abs_parity_error": core_max_error,
        "feature_columns": list(ENSEMBLE_ALL_COLS),
    }
    print(f"Fold {fold}: CORE parity={core_max_error:.2e}, "
          f"LambdaRank parity={lambda_max_error:.2e}, "
          f"TreeSHAP additivity={shap_error:.2e}; "
          f"exported {len(frame)} Core candidates")

manifest_path = EXPORT_ROOT_V30 / "manifest.json"
manifest_path.write_text(json.dumps(manifest_v30, indent=2), encoding="utf-8")
print("\nV30 explanation handoff written to:", EXPORT_ROOT_V30)
print("NOTE: Core family-level scores are exact; Core feature-level SHAP is NOT yet exported.")
print("NOTE: N5 v29 will NOT automatically read this new directory; N5 v30 needs an importer.")


Fold 0: CORE parity=0.00e+00, LambdaRank parity=0.00e+00, TreeSHAP additivity=2.22e-15; exported 120 Core candidates
Fold 1: CORE parity=0.00e+00, LambdaRank parity=0.00e+00, TreeSHAP additivity=4.44e-15; exported 120 Core candidates

V30 explanation handoff written to: /Users/neilyejjey/stock_signal_engine_v2/Notebooks/experiment_H_finalist_handoff/explainability_v30_handoff
NOTE: Core family-level scores are exact; Core feature-level SHAP is NOT yet exported.
NOTE: N5 v29 will NOT automatically read this new directory; N5 v30 needs an importer.
